# CTDDG pretraining — toxicity-aware

Original pretraining blocks 1–15, in original execution order. Run in the same project environment as before. The separate diagnostics notebook contains the original Block 16.

## Environment and Configuration

In [2]:
import os
import sys
import json
import time
import random
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import torch
import rdkit
from rdkit import Chem
import networkx as nx
import scipy

# ============================================================
# 1. PROJECT PATHS
# ============================================================

ROOT = Path("/home/muhamed/repo/CTDDG/REWORKED_CODE")

PREPROCESSING_DIR = ROOT / "PREPROCESSING_OUTPUT"
PRETRAINING_DIR = ROOT / "PRETRAINING_OUTPUT"

CHEMBL_FILE = (
    PREPROCESSING_DIR / "chembl" / "chembl_final.txt"
)

ATOM_TYPES_FILE = (
    PREPROCESSING_DIR / "vocabulary" / "atom_types.txt"
)

# ============================================================
# 2. EXPERIMENT CONFIGURATION
# ============================================================

SEED = 42

# Toxicity objective (Equation 8)
TOX_LAMBDA = 1.0

# Begin with a small pilot, not full training
TRAINING_STAGE = "pilot"
PILOT_STEPS = 20

# Unique directory for this lambda and training stage
LAMBDA_NAME = f"lambda_{TOX_LAMBDA:.2f}"

RUN_DIR = (
    PRETRAINING_DIR
    / "toxicity_aware"
    / LAMBDA_NAME
    / TRAINING_STAGE
)

RUN_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# 3. MODEL ARCHITECTURE
# ============================================================

N_ATOM = 55
N_BOND = 4

F_E = 16
F_H = [32, 64, 128, 128, 256, 256]
F_SKIP = 256
F_C = [512]
F_POLICY = 128

N_RNN = 3
RECEPTIVE_FIELD = 2

MAX_ITER_PER_MOL = 120

# ============================================================
# 4. GRAPH PATH SAMPLING
# ============================================================

K_PATHS = 5
ALPHA = 0.8

# ============================================================
# 5. TRAINING HYPERPARAMETERS
# ============================================================

BATCH_SIZE = 8

# Provisional full-training settings.
# We will revisit these after pilot measurements.
EPOCHS = 5

LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.0

LR_DECAY = 1e-3
LR_DECAY_STEP = 100

GRAD_CLIP = 3.0

LOG_EVERY = 100
SAVE_EVERY = 500

# New training run: never load an old checkpoint
RESUME = True

# ============================================================
# 6. VALIDATION CONFIGURATION
# ============================================================

VAL_FRACTION = 0.05
VAL_SEED = 2026

VAL_MAX_EVAL_MOLECULES = 512
VAL_EVERY = 5000

# ============================================================
# 7. DEVICE SETUP
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ============================================================
# 8. VALIDATE CONFIGURATION AND INPUT FILES
# ============================================================

assert TOX_LAMBDA >= 0
assert BATCH_SIZE >= 1
assert K_PATHS >= 1
assert 0 < ALPHA <= 1
assert 0 < VAL_FRACTION < 1
assert N_ATOM == 55
assert N_BOND == 4

for path in [CHEMBL_FILE, ATOM_TYPES_FILE]:
    if not path.is_file():
        raise FileNotFoundError(
            f"Required preprocessing output missing: {path}"
        )

# Verify the vocabulary is the exact one
# generated during our preprocessing.
with ATOM_TYPES_FILE.open("r", encoding="utf-8") as f:
    atom_types = [
        line.strip()
        for line in f
        if line.strip()
    ]

assert len(atom_types) == N_ATOM, (
    f"Expected {N_ATOM} atom types, "
    f"found {len(atom_types)}"
)

vocab_sha256 = hashlib.sha256(
    ATOM_TYPES_FILE.read_bytes()
).hexdigest()

EXPECTED_VOCAB_SHA256 = (
    "3f2a10321e1301f7a45f6ebb260181c70c0e28ae5c3162bd5c3e713db335184f"
)

assert vocab_sha256 == EXPECTED_VOCAB_SHA256, (
    "Atom vocabulary differs from the verified "
    "preprocessing vocabulary!"
)

# Quick input format check
with CHEMBL_FILE.open("r", encoding="utf-8") as f:
    for i in range(3):
        line = f.readline().strip()
        parts = line.rsplit(maxsplit=1)

        assert len(parts) == 2, (
            f"Invalid molecule format at line {i+1}"
        )
        assert parts[1] in {"0", "1"}, (
            f"Invalid Ames label at line {i+1}"
        )

# ============================================================
# 9. PRINT ENVIRONMENT SUMMARY
# ============================================================

print("=" * 65)
print("CTDDG TOXICITY-AWARE PRETRAINING — BLOCK 1")
print("=" * 65)

print("\n[ENVIRONMENT]")
print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("PyTorch:", torch.__version__)
print("RDKit:", rdkit.__version__)
print("NumPy:", np.__version__)
print("NetworkX:", nx.__version__)
print("SciPy:", scipy.__version__)

print("\n[GPU]")
print("Device:", DEVICE)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)

        print(f"GPU {i}: {props.name}")
        print(
            "Total VRAM:",
            f"{props.total_memory / (1024 ** 3):.2f} GB"
        )
else:
    print(
        "WARNING: No CUDA GPU detected. "
        "Full training may be very slow."
    )

print("\n[DATASET]")
print("ChEMBL:", CHEMBL_FILE)
print("Atom vocabulary:", ATOM_TYPES_FILE)
print("Atom vocabulary verified:", len(atom_types))
print("Vocabulary SHA256:", vocab_sha256)

print("\n[MODEL]")
print("Atom types:", N_ATOM)
print("Bond types:", N_BOND)
print("GCN layers:", F_H)
print("GRU layers:", N_RNN)
print("Importance paths:", K_PATHS)
print("Alpha:", ALPHA)

print("\n[TRAINING]")
print("Objective: Toxicity-aware Equation 8")
print("Lambda:", TOX_LAMBDA)
print("Batch size:", BATCH_SIZE)
print("Learning rate:", LEARNING_RATE)
print("Stage:", TRAINING_STAGE)
print("Pilot steps:", PILOT_STEPS)
print("Seed:", SEED)

print("\n[OUTPUT]")
print("Run directory:", RUN_DIR)

print("\n" + "=" * 65)
print("BLOCK 1 COMPLETED SUCCESSFULLY!")
print("=" * 65)


CTDDG TOXICITY-AWARE PRETRAINING — BLOCK 1

[ENVIRONMENT]
Python: 3.11.16
Executable: /home/muhamed/.conda/envs/ctddg_torch/bin/python
PyTorch: 2.4.1+cu121
RDKit: 2026.03.6
NumPy: 2.4.6
NetworkX: 3.6.1
SciPy: 1.17.1

[GPU]
Device: cuda
CUDA available: True
GPU 0: NVIDIA L40
Total VRAM: 44.37 GB
GPU 1: NVIDIA L40
Total VRAM: 44.37 GB

[DATASET]
ChEMBL: /home/muhamed/repo/CTDDG/REWORKED_CODE/PREPROCESSING_OUTPUT/chembl/chembl_final.txt
Atom vocabulary: /home/muhamed/repo/CTDDG/REWORKED_CODE/PREPROCESSING_OUTPUT/vocabulary/atom_types.txt
Atom vocabulary verified: 55
Vocabulary SHA256: 3f2a10321e1301f7a45f6ebb260181c70c0e28ae5c3162bd5c3e713db335184f

[MODEL]
Atom types: 55
Bond types: 4
GCN layers: [32, 64, 128, 128, 256, 256]
GRU layers: 3
Importance paths: 5
Alpha: 0.8

[TRAINING]
Objective: Toxicity-aware Equation 8
Lambda: 1.0
Batch size: 8
Learning rate: 0.001
Stage: pilot
Pilot steps: 20
Seed: 42

[OUTPUT]
Run directory: /home/muhamed/repo/CTDDG/REWORKED_CODE/PRETRAINING_OUTPUT/toxic

## Load ChEMBL and atom vocabulary

In [3]:

# ============================================================
# BLOCK 2: LOAD CHEMBL DATASET AND ATOM VOCABULARY
# ============================================================

from collections import Counter
from rdkit import Chem

# ------------------------------------------------------------
# 1. Atom and bond vocabulary
# ------------------------------------------------------------

class MoleculeSpec:

    def __init__(self, file_name):

        self.atom_types = []

        with open(file_name, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue

                parts = line.strip().split(",")

                if len(parts) != 3:
                    raise ValueError(
                        f"Invalid atom vocabulary entry: {line}"
                    )

                symbol = parts[0]
                charge = int(parts[1])
                explicit_hs = int(parts[2])

                self.atom_types.append(
                    (symbol, charge, explicit_hs)
                )

        if len(set(self.atom_types)) != len(self.atom_types):
            raise ValueError("Duplicate atom vocabulary entries")

        self.atom_index = {
            atom_type: idx
            for idx, atom_type in enumerate(self.atom_types)
        }

        self.bond_orders = [
            Chem.BondType.AROMATIC,
            Chem.BondType.SINGLE,
            Chem.BondType.DOUBLE,
            Chem.BondType.TRIPLE
        ]

        self.bond_index = {
            bond_type: idx
            for idx, bond_type in enumerate(self.bond_orders)
        }

        self.num_atom_types = len(self.atom_types)
        self.num_bond_types = len(self.bond_orders)

        self.max_iter = MAX_ITER_PER_MOL

    def get_atom_type(self, atom):

        key = (
            atom.GetSymbol(),
            atom.GetFormalCharge(),
            atom.GetNumExplicitHs()
        )

        if key not in self.atom_index:
            raise ValueError(
                f"Unknown atom type: {key}"
            )

        return self.atom_index[key]

    def get_bond_type(self, bond):

        bond_type = bond.GetBondType()

        if bond_type not in self.bond_index:
            raise ValueError(
                f"Unsupported bond type: {bond_type}"
            )

        return self.bond_index[bond_type]

    def index_to_atom(self, idx):

        symbol, charge, explicit_hs = self.atom_types[int(idx)]

        atom = Chem.Atom(symbol)
        atom.SetFormalCharge(charge)
        atom.SetNumExplicitHs(explicit_hs)

        return atom

    def index_to_bond(self, mol, begin_idx, end_idx, idx):

        mol.AddBond(
            int(begin_idx),
            int(end_idx),
            self.bond_orders[int(idx)]
        )


_mol_spec = None

def get_mol_spec():

    global _mol_spec

    if _mol_spec is None:
        _mol_spec = MoleculeSpec(ATOM_TYPES_FILE)

    return _mol_spec


# ------------------------------------------------------------
# 2. Load labelled ChEMBL
# ------------------------------------------------------------

def read_data(file_name):

    records = []
    label_counts = Counter()

    with open(file_name, "r", encoding="utf-8") as f:

        for line_number, line in enumerate(f, start=1):

            line = line.strip()

            if not line:
                continue

            parts = line.rsplit(maxsplit=1)

            if len(parts) != 2:
                raise ValueError(
                    f"Invalid record at line {line_number}"
                )

            smiles, label = parts

            if not smiles or label not in {"0", "1"}:
                raise ValueError(
                    f"Invalid SMILES/label at line {line_number}"
                )

            records.append(line)
            label_counts[int(label)] += 1

    return records, label_counts


dataset, label_counts = read_data(CHEMBL_FILE)

mol_spec = get_mol_spec()


# ------------------------------------------------------------
# 3. Integrity checks
# ------------------------------------------------------------

EXPECTED_MOLECULES = 1_090_506

assert len(dataset) == EXPECTED_MOLECULES, (
    f"Expected {EXPECTED_MOLECULES:,} molecules, "
    f"found {len(dataset):,}"
)

assert mol_spec.num_atom_types == N_ATOM
assert mol_spec.num_bond_types == N_BOND

assert sum(label_counts.values()) == len(dataset)

assert label_counts[0] > 0
assert label_counts[1] > 0


# ------------------------------------------------------------
# 4. Summary
# ------------------------------------------------------------

print("=" * 60)
print("BLOCK 2: DATASET AND VOCABULARY")
print("=" * 60)

print("\n[CHEMBL DATASET]")
print(f"Total molecules: {len(dataset):,}")
print(f"Ames-negative (0): {label_counts[0]:,}")
print(f"Ames-positive (1): {label_counts[1]:,}")

print("\n[ATOM VOCABULARY]")
print("Atom types:", mol_spec.num_atom_types)
print("Bond types:", mol_spec.num_bond_types)
print("Maximum graph steps:", mol_spec.max_iter)

print("\n[SAMPLE RECORDS]")
for record in dataset[:3]:
    print(record)

print("\nBLOCK 2 COMPLETED SUCCESSFULLY!")


BLOCK 2: DATASET AND VOCABULARY

[CHEMBL DATASET]
Total molecules: 1,090,506
Ames-negative (0): 781,280
Ames-positive (1): 309,226

[ATOM VOCABULARY]
Atom types: 55
Bond types: 4
Maximum graph steps: 120

[SAMPLE RECORDS]
Cc1ccc(CCN2CC(C(=O)NCc3cccnc3)CC2=O)cc1 0
O=C1NCCN1c1ccc(S(=O)(=O)Oc2ccc(F)cc2)cc1 0
CCOC(=O)c1c(C)oc2nc(C)nc(NCC3CCN(Cc4ccc(F)cc4)CC3)c12 1

BLOCK 2 COMPLETED SUCCESSFULLY!


## Reproducible Train/Validation Split


In [4]:

# ============================================================
# BLOCK 3: REPRODUCIBLE STRATIFIED TRAIN/VAL SPLIT
# ============================================================

import json
import hashlib
from pathlib import Path
import numpy as np

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

VAL_FRACTION = 0.05
VAL_SEED = 2026
VAL_MAX_EVAL_MOLECULES = 512

SPLIT_DIR = PRETRAINING_DIR / "shared"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_FILE = SPLIT_DIR / "chembl_split.npz"
SPLIT_MANIFEST = SPLIT_DIR / "chembl_split_manifest.json"

# ------------------------------------------------------------
# 2. Extract molecule labels
# ------------------------------------------------------------

labels = np.fromiter(
    (int(record.rsplit(maxsplit=1)[1]) for record in dataset),
    dtype=np.uint8,
    count=len(dataset)
)

assert len(labels) == len(dataset)
assert np.all((labels == 0) | (labels == 1))

# ------------------------------------------------------------
# 3. Hash input dataset for reproducibility
# ------------------------------------------------------------

def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


chembl_hash = sha256_file(CHEMBL_FILE)

# ------------------------------------------------------------
# 4. Create stratified train/validation indices
# ------------------------------------------------------------

rng = np.random.default_rng(VAL_SEED)

train_parts = []
val_parts = []

for label in [0, 1]:

    indices = np.flatnonzero(labels == label)
    rng.shuffle(indices)

    n_val = round(len(indices) * VAL_FRACTION)

    assert 0 < n_val < len(indices)

    val_parts.append(indices[:n_val])
    train_parts.append(indices[n_val:])

train_indices = np.sort(
    np.concatenate(train_parts)
)

val_indices = np.sort(
    np.concatenate(val_parts)
)

# ------------------------------------------------------------
# 5. Fixed validation subset for quick evaluation
# ------------------------------------------------------------

# Maintain approximate class proportions.
# Same subset will be used across lambda experiments.

rng_eval = np.random.default_rng(VAL_SEED + 1)

fixed_parts = []

for label in [0, 1]:

    class_val = val_indices[labels[val_indices] == label]

    rng_eval.shuffle(class_val)

    n_class = round(
        VAL_MAX_EVAL_MOLECULES
        * len(class_val)
        / len(val_indices)
    )

    fixed_parts.append(class_val[:n_class])

val_eval_indices = np.sort(
    np.concatenate(fixed_parts)
)

# ------------------------------------------------------------
# 6. Verify split integrity
# ------------------------------------------------------------

N = len(dataset)

assert len(train_indices) + len(val_indices) == N

assert len(np.unique(train_indices)) == len(train_indices)
assert len(np.unique(val_indices)) == len(val_indices)

assert np.all(train_indices >= 0)
assert np.all(val_indices >= 0)

assert train_indices.max() < N
assert val_indices.max() < N

train_mask = np.zeros(N, dtype=bool)
train_mask[train_indices] = True

assert not np.any(train_mask[val_indices])

assert np.all(
    np.isin(val_eval_indices, val_indices)
)

assert len(val_eval_indices) <= VAL_MAX_EVAL_MOLECULES

# ------------------------------------------------------------
# 7. Save split and metadata
# ------------------------------------------------------------

split_metadata = {
    "dataset_file": str(CHEMBL_FILE),
    "dataset_sha256": chembl_hash,
    "total_molecules": N,
    "train_molecules": len(train_indices),
    "validation_molecules": len(val_indices),
    "fixed_validation_molecules": len(val_eval_indices),
    "validation_fraction": VAL_FRACTION,
    "validation_seed": VAL_SEED,
    "fixed_validation_seed": VAL_SEED + 1,
    "train_class_0": int(np.sum(labels[train_indices] == 0)),
    "train_class_1": int(np.sum(labels[train_indices] == 1)),
    "val_class_0": int(np.sum(labels[val_indices] == 0)),
    "val_class_1": int(np.sum(labels[val_indices] == 1)),
    "split_file": str(SPLIT_FILE),
}

if SPLIT_FILE.exists() or SPLIT_MANIFEST.exists():

    # Never silently overwrite an existing experiment split.
    assert SPLIT_FILE.exists() and SPLIT_MANIFEST.exists(), (
        "Only one split output exists; inspect before continuing."
    )

    with np.load(SPLIT_FILE) as previous:
        assert np.array_equal(
            previous["train_indices"], train_indices
        )
        assert np.array_equal(
            previous["val_indices"], val_indices
        )
        assert np.array_equal(
            previous["val_eval_indices"], val_eval_indices
        )

    previous_metadata = json.loads(
        SPLIT_MANIFEST.read_text(encoding="utf-8")
    )

    assert previous_metadata == split_metadata, (
        "Saved split metadata differs from current data/config."
    )

    print("Existing matching split verified.")

else:

    with SPLIT_FILE.open("wb") as f:
        np.savez_compressed(
            f,
            train_indices=train_indices,
            val_indices=val_indices,
            val_eval_indices=val_eval_indices
        )

    SPLIT_MANIFEST.write_text(
        json.dumps(split_metadata, indent=2),
        encoding="utf-8"
    )

    print("New split saved successfully.")

# ------------------------------------------------------------
# 8. Display summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("BLOCK 3: STRATIFIED TRAIN/VALIDATION SPLIT")
print("=" * 65)

print(f"\nTotal molecules:       {N:,}")
print(f"Training molecules:    {len(train_indices):,}")
print(f"Validation molecules:  {len(val_indices):,}")
print(f"Fixed val subset:      {len(val_eval_indices):,}")

print("\nClass distribution:")

for name, indices in [
    ("Training", train_indices),
    ("Validation", val_indices),
    ("Fixed validation", val_eval_indices)
]:
    n0 = int(np.sum(labels[indices] == 0))
    n1 = int(np.sum(labels[indices] == 1))
    positive_pct = 100 * n1 / len(indices)

    print(
        f"{name:18s} | "
        f"Ames 0: {n0:>8,} | "
        f"Ames 1: {n1:>8,} | "
        f"Positive: {positive_pct:.2f}%"
    )

print("\nTrain/validation overlap: 0")

print("\nSaved files:")
print(SPLIT_FILE)
print(SPLIT_MANIFEST)

print("\nBLOCK 3 COMPLETED SUCCESSFULLY!")


Existing matching split verified.

BLOCK 3: STRATIFIED TRAIN/VALIDATION SPLIT

Total molecules:       1,090,506
Training molecules:    1,035,981
Validation molecules:  54,525
Fixed val subset:      512

Class distribution:
Training           | Ames 0:  742,216 | Ames 1:  293,765 | Positive: 28.36%
Validation         | Ames 0:   39,064 | Ames 1:   15,461 | Positive: 28.36%
Fixed validation   | Ames 0:      367 | Ames 1:      145 | Positive: 28.32%

Train/validation overlap: 0

Saved files:
/home/muhamed/repo/CTDDG/REWORKED_CODE/PRETRAINING_OUTPUT/shared/chembl_split.npz
/home/muhamed/repo/CTDDG/REWORKED_CODE/PRETRAINING_OUTPUT/shared/chembl_split_manifest.json

BLOCK 3 COMPLETED SUCCESSFULLY!


## Full Molecular Graph Compatibility Audit

In [5]:

# ============================================================
# BLOCK 4: FULL MOLECULAR GRAPH COMPATIBILITY AUDIT
# ============================================================

import json
import time
from collections import Counter, defaultdict

import numpy as np
from rdkit import Chem

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

AUDIT_DIR = PRETRAINING_DIR / "shared"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

AUDIT_FILE = AUDIT_DIR / "full_graph_audit.json"
ELIGIBLE_FILE = AUDIT_DIR / "graph_eligible_indices.npz"

MAX_EXAMPLES_PER_REASON = 5
PROGRESS_EVERY = 100_000

N = len(dataset)

print("=" * 65)
print("BLOCK 4: FULL MOLECULAR GRAPH COMPATIBILITY AUDIT")
print("=" * 65)

print(f"\nTotal molecules to check: {N:,}")
print(f"Atom vocabulary size: {mol_spec.num_atom_types}")
print(f"Bond types: {mol_spec.num_bond_types}")
print(f"Maximum graph steps: {mol_spec.max_iter}")

# ------------------------------------------------------------
# 2. Prepare audit counters
# ------------------------------------------------------------

valid_mask = np.zeros(N, dtype=bool)

reason_counts = Counter()
failure_examples = defaultdict(list)

max_atoms = 0
max_bonds = 0
max_graph_steps = 0

stereochemistry_count = 0
isotope_count = 0
radical_count = 0

start_time = time.perf_counter()

# ------------------------------------------------------------
# 3. Audit every molecule
# ------------------------------------------------------------

for idx, record in enumerate(dataset):

    smiles = record.rsplit(maxsplit=1)[0]

    reason = None

    try:
        mol = Chem.MolFromSmiles(smiles)
    except Exception:
        mol = None

    if mol is None:
        reason = "invalid_smiles"

    else:
        n_atoms = mol.GetNumAtoms()
        n_bonds = mol.GetNumBonds()

        # The reference graph path uses one step per
        # bond plus one initial step.
        graph_steps = n_bonds + 1

        max_atoms = max(max_atoms, n_atoms)
        max_bonds = max(max_bonds, n_bonds)
        max_graph_steps = max(
            max_graph_steps, graph_steps
        )

        # Record information the original graph
        # representation may not fully preserve.
        has_stereo = any(
            atom.GetChiralTag()
            != Chem.ChiralType.CHI_UNSPECIFIED
            for atom in mol.GetAtoms()
        ) or any(
            bond.GetStereo()
            != Chem.BondStereo.STEREONONE
            for bond in mol.GetBonds()
        )

        if has_stereo:
            stereochemistry_count += 1

        if any(
            atom.GetIsotope() != 0
            for atom in mol.GetAtoms()
        ):
            isotope_count += 1

        if any(
            atom.GetNumRadicalElectrons() > 0
            for atom in mol.GetAtoms()
        ):
            radical_count += 1

        # Structural compatibility checks
        if n_atoms < 2 or n_bonds < 1:
            reason = "empty_or_bondless_graph"

        elif len(Chem.GetMolFrags(mol)) != 1:
            reason = "disconnected_graph"

        elif graph_steps > mol_spec.max_iter:
            reason = "exceeds_max_graph_steps"

        else:
            # Check atom vocabulary
            for atom in mol.GetAtoms():

                atom_key = (
                    atom.GetSymbol(),
                    atom.GetFormalCharge(),
                    atom.GetNumExplicitHs()
                )

                if atom_key not in mol_spec.atom_index:
                    reason = "unknown_atom_type"
                    break

            # Check bond vocabulary
            if reason is None:

                for bond in mol.GetBonds():

                    if (
                        bond.GetBondType()
                        not in mol_spec.bond_index
                    ):
                        reason = "unsupported_bond_type"
                        break

    # Record final compatibility result
    if reason is None:
        valid_mask[idx] = True

    else:
        reason_counts[reason] += 1

        if (
            len(failure_examples[reason])
            < MAX_EXAMPLES_PER_REASON
        ):
            failure_examples[reason].append({
                "index": idx,
                "smiles": smiles
            })

    if (idx + 1) % PROGRESS_EVERY == 0:
        elapsed = time.perf_counter() - start_time

        print(
            f"Checked {idx + 1:,}/{N:,} | "
            f"Compatible: {int(valid_mask[:idx+1].sum()):,} | "
            f"Excluded: {sum(reason_counts.values()):,} | "
            f"Elapsed: {elapsed / 60:.1f} min"
        )

# ------------------------------------------------------------
# 4. Prepare compatible train/validation indices
# ------------------------------------------------------------

eligible_train_indices = train_indices[
    valid_mask[train_indices]
]

eligible_val_indices = val_indices[
    valid_mask[val_indices]
]

eligible_val_eval_indices = val_eval_indices[
    valid_mask[val_eval_indices]
]

excluded_indices = np.flatnonzero(~valid_mask)

total_eligible = int(valid_mask.sum())
total_excluded = N - total_eligible

# ------------------------------------------------------------
# 5. Integrity checks
# ------------------------------------------------------------

assert total_eligible + total_excluded == N

assert (
    len(eligible_train_indices)
    + len(eligible_val_indices)
    == total_eligible
)

assert len(eligible_train_indices) > 0
assert len(eligible_val_indices) > 0
assert len(eligible_val_eval_indices) > 0

assert np.all(
    np.isin(
        eligible_val_eval_indices,
        eligible_val_indices
    )
)

assert not np.any(
    np.isin(
        eligible_train_indices,
        eligible_val_indices
    )
)

# ------------------------------------------------------------
# 6. Build audit report
# ------------------------------------------------------------

elapsed_seconds = time.perf_counter() - start_time

audit_report = {
    "audit_name": "CTDDG_FULL_CHEMBL_GRAPH_AUDIT",
    "dataset_file": str(CHEMBL_FILE),
    "total_molecules": N,
    "compatible_molecules": total_eligible,
    "excluded_molecules": total_excluded,
    "exclusion_reasons": dict(reason_counts),
    "failure_examples": dict(failure_examples),
    "max_atoms_observed": max_atoms,
    "max_bonds_observed": max_bonds,
    "max_graph_steps_observed": max_graph_steps,
    "max_allowed_graph_steps": mol_spec.max_iter,
    "atom_vocabulary_size": mol_spec.num_atom_types,
    "bond_vocabulary_size": mol_spec.num_bond_types,
    "molecules_with_stereochemistry": (
        stereochemistry_count
    ),
    "molecules_with_isotopes": isotope_count,
    "molecules_with_radicals": radical_count,
    "eligible_training_molecules": int(
        len(eligible_train_indices)
    ),
    "eligible_validation_molecules": int(
        len(eligible_val_indices)
    ),
    "eligible_fixed_validation_molecules": int(
        len(eligible_val_eval_indices)
    ),
    "full_graph_traversal_tested": False,
    "notes": [
        "Original ChEMBL dataset is unchanged.",
        "Eligibility uses atom/bond, connectivity and "
        "graph step checks.",
        "Stereochemistry is recorded but not excluded.",
        "Isotope and radical information may not be "
        "represented by the reference graph encoder.",
        "Full traversal and batching will be tested "
        "in subsequent blocks."
    ]
}

# ------------------------------------------------------------
# 7. Save outputs without overwriting previous results
# ------------------------------------------------------------

if AUDIT_FILE.exists() or ELIGIBLE_FILE.exists():

    assert AUDIT_FILE.exists() and ELIGIBLE_FILE.exists(), (
        "Only one audit output exists. "
        "Inspect the saved files before continuing."
    )

    old_report = json.loads(
        AUDIT_FILE.read_text(encoding="utf-8")
    )

    # Elapsed time is not part of the saved report,
    # so results can be compared reproducibly.
    assert old_report == audit_report, (
        "Existing audit differs from this run."
    )

    with np.load(ELIGIBLE_FILE) as previous:

        assert np.array_equal(
            previous["train_indices"],
            eligible_train_indices
        )

        assert np.array_equal(
            previous["val_indices"],
            eligible_val_indices
        )

        assert np.array_equal(
            previous["val_eval_indices"],
            eligible_val_eval_indices
        )

        assert np.array_equal(
            previous["excluded_indices"],
            excluded_indices
        )

    print("\nExisting audit verified successfully.")

else:

    np.savez_compressed(
        ELIGIBLE_FILE,
        train_indices=eligible_train_indices,
        val_indices=eligible_val_indices,
        val_eval_indices=eligible_val_eval_indices,
        excluded_indices=excluded_indices
    )

    AUDIT_FILE.write_text(
        json.dumps(audit_report, indent=2),
        encoding="utf-8"
    )

    print("\nAudit files saved successfully.")

# ------------------------------------------------------------
# 8. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FULL GRAPH COMPATIBILITY SUMMARY")
print("=" * 65)

print(f"Total molecules:      {N:,}")
print(f"Compatible molecules: {total_eligible:,}")
print(f"Excluded molecules:   {total_excluded:,}")

print("\nExclusion reasons:")

if reason_counts:
    for reason, count in reason_counts.items():
        print(f"  {reason}: {count:,}")
else:
    print("  None")

print("\nGraph statistics:")
print("Maximum atoms:", max_atoms)
print("Maximum bonds:", max_bonds)
print("Maximum graph steps:", max_graph_steps)
print("Allowed graph steps:", mol_spec.max_iter)

print("\nRepresentation diagnostics:")
print("Molecules with stereochemistry:", stereochemistry_count)
print("Molecules with isotopes:", isotope_count)
print("Molecules with radicals:", radical_count)

print("\nEligible dataset splits:")
print(
    "Training:",
    f"{len(eligible_train_indices):,}"
)
print(
    "Validation:",
    f"{len(eligible_val_indices):,}"
)
print(
    "Fixed validation:",
    f"{len(eligible_val_eval_indices):,}"
)

print("\nSaved outputs:")
print(AUDIT_FILE)
print(ELIGIBLE_FILE)

print(
    f"\nElapsed time: {elapsed_seconds / 60:.2f} minutes"
)

print("\nBLOCK 4 COMPLETED SUCCESSFULLY!")


BLOCK 4: FULL MOLECULAR GRAPH COMPATIBILITY AUDIT

Total molecules to check: 1,090,506
Atom vocabulary size: 55
Bond types: 4
Maximum graph steps: 120
Checked 100,000/1,090,506 | Compatible: 100,000 | Excluded: 0 | Elapsed: 0.4 min
Checked 200,000/1,090,506 | Compatible: 200,000 | Excluded: 0 | Elapsed: 0.8 min
Checked 300,000/1,090,506 | Compatible: 300,000 | Excluded: 0 | Elapsed: 1.2 min
Checked 400,000/1,090,506 | Compatible: 400,000 | Excluded: 0 | Elapsed: 1.6 min
Checked 500,000/1,090,506 | Compatible: 500,000 | Excluded: 0 | Elapsed: 2.0 min
Checked 600,000/1,090,506 | Compatible: 600,000 | Excluded: 0 | Elapsed: 2.4 min
Checked 700,000/1,090,506 | Compatible: 700,000 | Excluded: 0 | Elapsed: 2.8 min
Checked 800,000/1,090,506 | Compatible: 800,000 | Excluded: 0 | Elapsed: 3.2 min
Checked 900,000/1,090,506 | Compatible: 900,000 | Excluded: 0 | Elapsed: 3.6 min
Checked 1,000,000/1,090,506 | Compatible: 1,000,000 | Excluded: 0 | Elapsed: 4.0 min

Existing audit verified successful

## Molecular Graph Construction

In [6]:

# ============================================================
# BLOCK 5: MOLECULAR GRAPH CONSTRUCTION
# CTDDG / CDGCN REFERENCE GRAPH REPRESENTATION
# ============================================================

import numpy as np
import networkx as nx
from rdkit import Chem

# ------------------------------------------------------------
# 1. Convert a SMILES string into the CTDDG graph format
# ------------------------------------------------------------

def get_graph_from_smiles(smiles):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        raise ValueError(
            f"RDKit cannot parse SMILES: {smiles}"
        )

    if mol.GetNumAtoms() == 0:
        raise ValueError("Molecule contains no atoms")

    spec = get_mol_spec()

    atom_types = []
    atom_ranks = []
    bonds = []
    bond_types = []

    # Canonical ranks guide the reference graph traversal
    canonical_ranks = list(Chem.CanonicalRankAtoms(mol))

    # Encode atoms
    for atom, rank in zip(
        mol.GetAtoms(), canonical_ranks
    ):
        atom_types.append(
            spec.get_atom_type(atom)
        )

        atom_ranks.append(int(rank))

    # Encode bonds
    for bond in mol.GetBonds():

        begin_idx = bond.GetBeginAtomIdx()
        end_idx = bond.GetEndAtomIdx()

        bond_type = spec.get_bond_type(bond)

        bonds.append([
            begin_idx,
            end_idx
        ])

        bond_types.append(bond_type)

    # Build undirected NetworkX graph
    graph = nx.Graph()

    graph.add_nodes_from(
        range(len(atom_types))
    )

    graph.add_edges_from(bonds)

    if not nx.is_connected(graph):
        raise ValueError(
            f"Disconnected molecular graph: {smiles}"
        )

    if graph.number_of_edges() != len(bonds):
        raise ValueError(
            "Graph edge count does not match bond count"
        )

    return (
        graph,
        atom_types,
        atom_ranks,
        bonds,
        bond_types
    )


# ------------------------------------------------------------
# 2. Convert multiple SMILES to compact graph arrays
# ------------------------------------------------------------

def get_graph_from_smiles_list(smiles_list):

    graph_list = []

    for smiles in smiles_list:

        (
            graph,
            atom_types,
            atom_ranks,
            bonds,
            bond_types
        ) = get_graph_from_smiles(smiles)

        # X: atom type indices
        X = np.asarray(
            atom_types,
            dtype=np.int32
        )

        # A: [begin_atom, end_atom, bond_type]
        bond_array = np.asarray(
            bonds,
            dtype=np.int32
        ).reshape(-1, 2)

        bond_type_array = np.asarray(
            bond_types,
            dtype=np.int32
        ).reshape(-1, 1)

        A = np.concatenate(
            [bond_array, bond_type_array],
            axis=1
        )

        graph_list.append([X, A])

    return graph_list


# ------------------------------------------------------------
# 3. Test graph construction on real ChEMBL molecules
# ------------------------------------------------------------

print("=" * 65)
print("BLOCK 5: MOLECULAR GRAPH CONSTRUCTION")
print("=" * 65)

# Test molecules from both training and validation sets
test_indices = [
    int(train_indices[0]),
    int(train_indices[len(train_indices) // 4]),
    int(train_indices[len(train_indices) // 2]),
    int(train_indices[-1]),
    int(val_indices[0]),
    int(val_indices[len(val_indices) // 2]),
    int(val_indices[-1])
]

test_smiles = [
    dataset[idx].rsplit(maxsplit=1)[0]
    for idx in test_indices
]

total_atoms = 0
total_bonds = 0

for i, smiles in enumerate(test_smiles, start=1):

    (
        graph,
        atom_types,
        atom_ranks,
        bonds,
        bond_types
    ) = get_graph_from_smiles(smiles)

    mol = Chem.MolFromSmiles(smiles)

    # Verify node and edge consistency
    assert graph.number_of_nodes() == mol.GetNumAtoms()
    assert graph.number_of_edges() == mol.GetNumBonds()

    assert len(atom_types) == mol.GetNumAtoms()
    assert len(atom_ranks) == mol.GetNumAtoms()

    assert len(bonds) == mol.GetNumBonds()
    assert len(bond_types) == mol.GetNumBonds()

    # Verify valid vocabulary indices
    assert all(
        0 <= x < N_ATOM
        for x in atom_types
    )

    assert all(
        0 <= b < N_BOND
        for b in bond_types
    )

    assert nx.is_connected(graph)

    # Verify graph fits the model's step limit
    assert (
        graph.number_of_edges() + 1
        <= MAX_ITER_PER_MOL
    )

    total_atoms += graph.number_of_nodes()
    total_bonds += graph.number_of_edges()

    print(f"\nMolecule {i}")
    print("SMILES:", smiles[:100])
    print("Atoms:", graph.number_of_nodes())
    print("Bonds:", graph.number_of_edges())
    print("Atom IDs (first 10):", atom_types[:10])
    print("Canonical ranks (first 10):", atom_ranks[:10])


# ------------------------------------------------------------
# 4. Verify compact graph array format
# ------------------------------------------------------------

graph_list = get_graph_from_smiles_list(
    test_smiles
)

assert len(graph_list) == len(test_smiles)

for smiles, (X, A) in zip(
    test_smiles, graph_list
):

    mol = Chem.MolFromSmiles(smiles)

    assert X.ndim == 1
    assert A.ndim == 2
    assert A.shape[1] == 3

    assert len(X) == mol.GetNumAtoms()
    assert len(A) == mol.GetNumBonds()

    assert np.all(X >= 0)
    assert np.all(X < N_ATOM)

    assert np.all(A[:, :2] >= 0)
    assert np.all(A[:, :2] < len(X))

    assert np.all(A[:, 2] >= 0)
    assert np.all(A[:, 2] < N_BOND)

print("\n" + "=" * 65)
print("GRAPH CONSTRUCTION SUMMARY")
print("=" * 65)

print("Tested molecules:", len(test_smiles))
print("Total atoms represented:", total_atoms)
print("Total bonds represented:", total_bonds)
print("Atom vocabulary:", N_ATOM)
print("Bond vocabulary:", N_BOND)

print("\nGraph connectivity: PASSED")
print("Atom encoding: PASSED")
print("Bond encoding: PASSED")
print("Compact graph arrays: PASSED")
print("Graph step limits: PASSED")

print("\nBLOCK 5 COMPLETED SUCCESSFULLY!")


BLOCK 5: MOLECULAR GRAPH CONSTRUCTION

Molecule 1
SMILES: Cc1ccc(CCN2CC(C(=O)NCc3cccnc3)CC2=O)cc1
Atoms: 25
Bonds: 27
Atom IDs (first 10): [6, 6, 6, 6, 6, 6, 6, 24, 6, 6]
Canonical ranks (first 10): [0, 20, 5, 7, 21, 11, 14, 24, 15, 23]

Molecule 2
SMILES: Nc1ccc(-c2cc(C3CC4CCC3N4)cnc2F)cn1
Atoms: 21
Bonds: 24
Atom IDs (first 10): [24, 6, 6, 6, 6, 6, 6, 6, 6, 6]
Canonical ranks (first 10): [0, 15, 5, 2, 13, 16, 6, 14, 20, 9]

Molecule 3
SMILES: CN1[C@@H]2CC[C@H]1C[C@@H](OC(=O)C(O)c1ccccc1)C2
Atoms: 20
Bonds: 22
Atom IDs (first 10): [6, 24, 7, 6, 6, 7, 6, 7, 32, 6]
Canonical ranks (first 10): [0, 19, 17, 8, 9, 18, 11, 16, 12, 13]

Molecule 4
SMILES: COc1cc([C@@H]2c3cc4c(cc3[C@H](O)[C@H]3COC(=O)[C@@H]23)OCO4)cc(OC)c1OC
Atoms: 30
Bonds: 34
Atom IDs (first 10): [6, 32, 6, 6, 6, 7, 6, 6, 6, 6]
Canonical ranks (first 10): [0, 14, 21, 5, 18, 26, 23, 7, 19, 20]

Molecule 5
SMILES: Cn1c(SCc2ccc(C#N)cc2Cl)nc2ccccc21
Atoms: 21
Bonds: 23
Atom IDs (first 10): [6, 24, 6, 44, 6, 6, 6, 6, 6, 6]
Canoni

## Graph Traversal and Importance Sampling

In [7]:

# ============================================================
# BLOCK 6: GRAPH TRAVERSAL AND IMPORTANCE SAMPLING
# CTDDG / CDGCN REFERENCE IMPLEMENTATION
# ============================================================

import random
import numpy as np
import networkx as nx
from rdkit import Chem

# ------------------------------------------------------------
# 1. Stochastic graph traversal
# ------------------------------------------------------------

def traverse_graph(
    graph,
    atom_ranks,
    current_node=None,
    step_ids=None,
    p=ALPHA,
    log_p=0.0
):
    """
    Generate a stochastic depth-first traversal.

    Returns:
        step_ids: construction order assigned to each atom
        log_p: log probability of this traversal
               under the proposal distribution Q_alpha
    """

    if not 0.0 < p <= 1.0:
        raise ValueError("Traversal probability p must be in (0, 1]")

    if current_node is None:

        if not nx.is_connected(graph):
            raise ValueError("Graph must be connected")

        next_nodes = list(range(len(atom_ranks)))
        step_ids = [-1] * len(atom_ranks)

    else:

        next_nodes = [
            n for n in graph.neighbors(current_node)
            if step_ids[n] < 0
        ]

    # Prefer nodes with lower canonical ranks
    next_nodes = sorted(
        next_nodes,
        key=lambda n: atom_ranks[n]
    )

    while next_nodes:

        if len(next_nodes) == 1:

            next_node = next_nodes[0]

        elif random.random() >= (1.0 - p):

            # Choose the canonical-rank-preferred atom
            next_node = next_nodes[0]
            log_p += np.log(p)

        else:

            # Choose one of the other available atoms
            next_node = next_nodes[
                random.randint(1, len(next_nodes) - 1)
            ]

            log_p += np.log(
                (1.0 - p) / (len(next_nodes) - 1)
            )

        # Assign next construction step
        step_ids[next_node] = max(step_ids) + 1

        # Continue traversal from selected atom
        _, log_p = traverse_graph(
            graph,
            atom_ranks,
            current_node=next_node,
            step_ids=step_ids,
            p=p,
            log_p=log_p
        )

        # Remove already visited nodes
        next_nodes = [
            n for n in next_nodes
            if step_ids[n] < 0
        ]

    return step_ids, float(log_p)


# ------------------------------------------------------------
# 2. Reorder atoms and bonds using traversal order
# ------------------------------------------------------------

def single_reorder(X_0, A_0, step_ids):
    """
    Reorder a molecular graph according to a sampled path.

    X_0: atom type IDs, shape [num_atoms]
    A_0: bonds [begin, end, bond_type], shape [num_bonds, 3]
    step_ids: traversal position of each original atom

    Returns reordered X_0 and A_0.

    Bond orientation:
      begin < end : append a new atom
      begin > end : connect existing atoms
    """

    X_0 = np.asarray(X_0, dtype=np.int32).copy()

    A_0 = np.asarray(
        A_0, dtype=np.int32
    ).reshape(-1, 3).copy()

    step_ids = np.asarray(
        step_ids, dtype=np.int32
    )

    n_atoms = len(X_0)

    assert len(step_ids) == n_atoms
    assert np.array_equal(
        np.sort(step_ids),
        np.arange(n_atoms)
    )

    # Singleton support
    if len(A_0) == 0:

        if n_atoms != 1:
            raise ValueError(
                "Only a single atom may have no bonds"
            )

        return X_0, A_0

    # Reorder atom IDs
    sorted_ids = np.argsort(step_ids)
    X_0 = X_0[sorted_ids]

    # Map original bond endpoints to traversal indices
    A_0[:, 0], A_0[:, 1] = (
        step_ids[A_0[:, 0]],
        step_ids[A_0[:, 1]]
    )

    max_b = np.max(A_0[:, :2], axis=1)
    min_b = np.min(A_0[:, :2], axis=1)

    # Sort by the later atom, then the earlier atom
    # in descending order (reference algorithm).
    order = np.lexsort([-min_b, max_b])
    A_0 = A_0[order, :]

    max_b = np.max(A_0[:, :2], axis=1)
    min_b = np.min(A_0[:, :2], axis=1)

    # First bond introducing a new atom = append.
    # Additional bonds to that atom = connect.
    is_append = np.concatenate([
        np.array([True]),
        max_b[1:] > max_b[:-1]
    ])

    oriented_endpoints = np.where(
        is_append[:, None],
        np.stack([min_b, max_b], axis=1),
        np.stack([max_b, min_b], axis=1)
    )

    A_0 = np.concatenate([
        oriented_endpoints,
        A_0[:, -1:]
    ], axis=1).astype(np.int32)

    return X_0, A_0


# ------------------------------------------------------------
# 3. Sample K different construction paths
# ------------------------------------------------------------

def sample_graph_paths(smiles, k=K_PATHS, alpha=ALPHA):
    """
    Sample K graph construction paths for one molecule.
    """

    (
        graph,
        atom_types,
        atom_ranks,
        bonds,
        bond_types
    ) = get_graph_from_smiles(smiles)

    X_0 = np.asarray(atom_types, dtype=np.int32)

    A_0 = np.column_stack([
        np.asarray(bonds, dtype=np.int32).reshape(-1, 2),
        np.asarray(bond_types, dtype=np.int32)
    ])

    results = []

    for path_id in range(k):

        step_ids, log_p = traverse_graph(
            graph,
            atom_ranks,
            p=alpha
        )

        X, A = single_reorder(
            X_0, A_0, step_ids
        )

        results.append({
            "path_id": path_id,
            "step_ids": step_ids,
            "X": X,
            "A": A,
            "log_p": log_p
        })

    return results


# ------------------------------------------------------------
# 4. Test on real ChEMBL molecules
# ------------------------------------------------------------

print("=" * 65)
print("BLOCK 6: GRAPH TRAVERSAL AND IMPORTANCE SAMPLING")
print("=" * 65)

# Include ordinary molecules plus at least one ring molecule.
selected_smiles = [
    dataset[int(train_indices[0])].rsplit(maxsplit=1)[0],
    dataset[int(train_indices[len(train_indices)//2])].rsplit(maxsplit=1)[0]
]

ring_smiles = None

for idx in train_indices[:1000]:

    candidate = dataset[int(idx)].rsplit(maxsplit=1)[0]
    mol = Chem.MolFromSmiles(candidate)

    if mol is not None and mol.GetRingInfo().NumRings() > 0:
        ring_smiles = candidate
        break

if ring_smiles is None:
    raise RuntimeError(
        "Could not find a ring molecule for traversal testing"
    )

selected_smiles.append(ring_smiles)

tested_paths = 0
tested_connect_actions = 0

for mol_number, smiles in enumerate(selected_smiles, 1):

    graph, atom_types, atom_ranks, bonds, bond_types = (
        get_graph_from_smiles(smiles)
    )

    paths = sample_graph_paths(
        smiles,
        k=K_PATHS,
        alpha=ALPHA
    )

    assert len(paths) == K_PATHS

    print(f"\nMolecule {mol_number}")
    print("SMILES:", smiles[:110])
    print("Atoms:", len(atom_types))
    print("Bonds:", len(bonds))

    for path in paths:

        step_ids = path["step_ids"]
        X = path["X"]
        A = path["A"]
        log_p = path["log_p"]

        # Every atom visited exactly once
        assert sorted(step_ids) == list(
            range(len(atom_types))
        )

        # Atom and bond counts preserved
        assert len(X) == len(atom_types)
        assert len(A) == len(bonds)

        # Reordered atoms must match traversal positions
        assert np.array_equal(
            X,
            np.asarray(atom_types)[np.argsort(step_ids)]
        )

        # Verify original bonds are preserved
        original_bonds = sorted([
            (
                min(step_ids[a], step_ids[b]),
                max(step_ids[a], step_ids[b]),
                int(bt)
            )
            for (a, b), bt in zip(bonds, bond_types)
        ])

        reordered_bonds = sorted([
            (
                min(int(row[0]), int(row[1])),
                max(int(row[0]), int(row[1])),
                int(row[2])
            )
            for row in A
        ])

        assert original_bonds == reordered_bonds

        # Append actions introduce new atoms.
        append_mask = A[:, 0] < A[:, 1]
        connect_mask = A[:, 0] > A[:, 1]

        n_append = int(append_mask.sum())
        n_connect = int(connect_mask.sum())

        assert n_append == len(X) - 1
        assert n_connect == len(A) - len(X) + 1

        # Every non-initial atom is introduced once.
        assert sorted(A[append_mask, 1].tolist()) == (
            list(range(1, len(X)))
        )

        assert np.isfinite(log_p)
        assert log_p <= 1e-8

        tested_paths += 1
        tested_connect_actions += n_connect

        print(
            f"  Path {path['path_id'] + 1}: "
            f"append={n_append}, "
            f"connect={n_connect}, "
            f"log(Q)={log_p:.4f}"
        )

# ------------------------------------------------------------
# 5. Deterministic traversal sanity check
# ------------------------------------------------------------

g, atoms, ranks, bonds, bond_types = (
    get_graph_from_smiles(ring_smiles)
)

steps_a, log_a = traverse_graph(
    g, ranks, p=1.0
)

steps_b, log_b = traverse_graph(
    g, ranks, p=1.0
)

assert steps_a == steps_b
assert np.isclose(log_a, 0.0)
assert np.isclose(log_b, 0.0)

print("\n" + "=" * 65)
print("TRAVERSAL SUMMARY")
print("=" * 65)

print("Tested molecules:", len(selected_smiles))
print("Sampled paths:", tested_paths)
print("Total connect actions:", tested_connect_actions)
print("Importance paths per molecule:", K_PATHS)
print("Sampling alpha:", ALPHA)

print("\nAtom traversal permutations: PASSED")
print("Bond preservation: PASSED")
print("Append/connect classification: PASSED")
print("Log proposal probabilities: PASSED")
print("Deterministic traversal (p=1): PASSED")

print("\nBLOCK 6 COMPLETED SUCCESSFULLY!")


BLOCK 6: GRAPH TRAVERSAL AND IMPORTANCE SAMPLING

Molecule 1
SMILES: Cc1ccc(CCN2CC(C(=O)NCc3cccnc3)CC2=O)cc1
Atoms: 25
Bonds: 27
  Path 1: append=24, connect=3, log(Q)=-10.5084
  Path 2: append=24, connect=3, log(Q)=-2.9483
  Path 3: append=24, connect=3, log(Q)=-7.1072
  Path 4: append=24, connect=3, log(Q)=-1.5620
  Path 5: append=24, connect=3, log(Q)=-2.9483

Molecule 2
SMILES: CN1[C@@H]2CC[C@H]1C[C@@H](OC(=O)C(O)c1ccccc1)C2
Atoms: 20
Bonds: 22
  Path 1: append=19, connect=3, log(Q)=-2.9483
  Path 2: append=19, connect=3, log(Q)=-4.3346
  Path 3: append=19, connect=3, log(Q)=-2.9483
  Path 4: append=19, connect=3, log(Q)=-2.9483
  Path 5: append=19, connect=3, log(Q)=-6.1159

Molecule 3
SMILES: Cc1ccc(CCN2CC(C(=O)NCc3cccnc3)CC2=O)cc1
Atoms: 25
Bonds: 27
  Path 1: append=24, connect=3, log(Q)=-4.3346
  Path 2: append=24, connect=3, log(Q)=-7.7358
  Path 3: append=24, connect=3, log(Q)=-3.1714
  Path 4: append=24, connect=3, log(Q)=-7.9589
  Path 5: append=24, connect=3, log(Q)=-10.7

## Sequential Graph Expansion and Batch Construction

In [8]:

# ============================================================
# BLOCK 7: SEQUENTIAL GRAPH EXPANSION AND BATCH CONSTRUCTION
# CTDDG TOXICITY-AWARE PRETRAINING
# ============================================================

import numpy as np
import torch
from scipy import sparse
from rdkit import Chem

# ============================================================
# 1. EXPAND ONE REORDERED MOLECULAR GRAPH
# ============================================================

def single_expand(X_0, A_0):
    """
    Convert one reordered molecule into sequential graph states.

    Actions:
        0 = APPEND
        1 = CONNECT
        2 = END

    Returns:
        action_0, X, NX, A, NA, actions, last_atom_mask
    """

    X_0 = np.asarray(X_0, dtype=np.int32).copy()
    A_0 = np.asarray(A_0, dtype=np.int32).reshape(-1, 3).copy()

    n_atoms = len(X_0)
    n_bonds = len(A_0)

    if n_atoms < 1:
        raise ValueError("Empty molecule")

    if n_bonds == 0:
        raise ValueError(
            "Bondless molecules are not part of our "
            "audited ChEMBL training dataset."
        )

    is_append = (A_0[:, 0] < A_0[:, 1]).astype(np.int32)

    # Number of atoms in each intermediate graph
    NX = np.cumsum(
        np.pad(
            is_append,
            (1, 0),
            mode="constant",
            constant_values=1
        )
    ).astype(np.int32)

    assert len(NX) == n_bonds + 1
    assert NX[-1] == n_atoms

    shift = np.cumsum(
        np.pad(NX, (1, 0), mode="constant")[:-1]
    )

    X_index = (
        np.arange(NX.sum(), dtype=np.int32)
        - np.repeat(shift, NX)
    )

    # All intermediate graph atom arrays concatenated
    X = X_0[X_index]

    # Intermediate graph bond arrays
    _, A_index = np.tril_indices(n_bonds)

    A = A_0[A_index, :]

    NA = np.arange(n_bonds + 1, dtype=np.int32)

    # Construct action targets
    action_type = 1 - is_append

    atom_type = np.where(
        action_type == 0,
        X_0[A_0[:, 1]],
        0
    )

    bond_type = A_0[:, 2]

    append_pos = np.where(
        action_type == 0,
        A_0[:, 0],
        0
    )

    connect_pos = np.where(
        action_type == 1,
        A_0[:, 1],
        0
    )

    actions = np.stack(
        [
            action_type,
            atom_type,
            bond_type,
            append_pos,
            connect_pos
        ],
        axis=1
    ).astype(np.int32)

    # Final END action
    end_action = np.array(
        [[2, 0, 0, 0, 0]],
        dtype=np.int32
    )

    actions = np.concatenate(
        [actions, end_action],
        axis=0
    )

    # Initial atom choice
    action_0 = np.array(
        [X_0[0]],
        dtype=np.int32
    )

    # Mark the most recently added atom
    last_atom_index = shift + NX - 1

    last_atom_mask = np.zeros_like(X)

    append_flags = np.pad(
        is_append,
        (1, 0),
        mode="constant",
        constant_values=1
    )

    last_atom_mask[last_atom_index] = np.where(
        append_flags == 1,
        1,
        2
    )

    assert len(actions) == len(NX)
    assert len(X) == int(NX.sum())
    assert len(A) == int(NA.sum())
    assert len(last_atom_mask) == len(X)

    return (
        action_0,
        X,
        NX,
        A,
        NA,
        actions,
        last_atom_mask
    )


# ============================================================
# 2. COMPUTE GRAPH-DISTANCE RELATIONSHIPS
# ============================================================

def get_d(A, X):
    """
    Construct distance-2 and distance-3 relationships
    using the reference CDGCN sparse-matrix method.
    """

    A = np.asarray(A, dtype=np.int32).reshape(-1, 3)

    def to_sparse(edges, num_nodes):

        edges = np.asarray(
            edges, dtype=np.int32
        ).reshape(-1, 2)

        return sparse.coo_matrix(
            (
                np.ones(
                    len(edges) * 2,
                    dtype=np.int32
                ),
                (
                    np.concatenate([
                        edges[:, 0],
                        edges[:, 1]
                    ]),
                    np.concatenate([
                        edges[:, 1],
                        edges[:, 0]
                    ])
                )
            ),
            shape=(num_nodes, num_nodes)
        ).tocsr()

    n_nodes = len(X)

    adjacency = to_sparse(
        A[:, :2],
        n_nodes
    )

    d2 = adjacency @ adjacency
    d3 = d2 @ adjacency

    D_2 = np.stack(
        d2.nonzero(),
        axis=1
    ).astype(np.int32)

    D_2 = D_2[
        D_2[:, 0] < D_2[:, 1]
    ]

    D_3 = np.stack(
        d3.nonzero(),
        axis=1
    ).astype(np.int32)

    D_3 = D_3[
        D_3[:, 0] < D_3[:, 1]
    ]

    # Exclude directly bonded pairs from D3,
    # following the original reference implementation.
    D_3_sparse = to_sparse(D_3, n_nodes)

    D_3_sparse = (
        D_3_sparse
        - D_3_sparse.multiply(adjacency)
    )

    D_3 = np.stack(
        D_3_sparse.nonzero(),
        axis=1
    ).astype(np.int32)

    D_3 = D_3[
        D_3[:, 0] < D_3[:, 1]
    ]

    return D_2, D_3


# ============================================================
# 3. MERGE INTERMEDIATE GRAPH STATES
# ============================================================

def merge_single_0(X, A, NX, NA):

    X = np.asarray(X, dtype=np.int32)
    A = np.asarray(A, dtype=np.int32).reshape(-1, 3).copy()
    NX = np.asarray(NX, dtype=np.int32)
    NA = np.asarray(NA, dtype=np.int32)

    offsets = np.cumsum(
        np.pad(NX, (1, 0), mode="constant")[:-1]
    )

    # Convert local atom indices to global indices
    A[:, :2] += np.repeat(
        offsets, NA
    )[:, None]

    D_2, D_3 = get_d(A, X)

    # One adjacency list per bond type
    A_split = []

    for bond_id in range(
        get_mol_spec().num_bond_types
    ):

        A_i = A[
            A[:, 2] == bond_id,
            :2
        ]

        A_split.append(A_i)

    # Two graph-distance relations
    A_split.extend([D_2, D_3])

    # Map every atom to its intermediate graph
    NX_rep = np.repeat(
        np.arange(len(NX)),
        NX
    ).astype(np.int32)

    return X, A_split, NX, NX_rep


def merge_single(
    X, A, NX, NA,
    mol_ids, rep_ids, iw_ids,
    action_0, actions,
    last_atom_mask, log_p
):

    X, A, NX, NX_rep = merge_single_0(
        X, A, NX, NA
    )

    offsets = np.cumsum(
        np.pad(NX, (1, 0), mode="constant")[:-1]
    )

    actions = actions.copy()

    # Convert action positions to global node indices
    actions[:, 3] += offsets * (
        actions[:, 0] == 0
    )

    actions[:, 4] += offsets * (
        actions[:, 0] == 1
    )

    mol_ids_rep = np.repeat(
        mol_ids, NX
    )

    rep_ids_rep = np.repeat(
        rep_ids, NX
    )

    return (
        X, A,
        mol_ids_rep, rep_ids_rep,
        iw_ids, last_atom_mask,
        NX, NX_rep,
        action_0, actions, log_p
    )


# ============================================================
# 4. PROCESS ONE LABELLED MOLECULE INTO K PATHS
# ============================================================

def process_single(record, k=K_PATHS, p=ALPHA):
    """
    Input:
        '<SMILES> <AMES_LABEL>'

    Returns:
        K sampled graph paths,
        and exactly ONE molecular toxicity label.
    """

    parts = record.strip().rsplit(maxsplit=1)

    if len(parts) != 2:
        raise ValueError(
            f"Invalid labelled molecule: {record[:100]}"
        )

    smiles, label_raw = parts

    if label_raw not in {"0", "1"}:
        raise ValueError(
            f"Invalid Ames label: {label_raw}"
        )

    toxicity_label = int(label_raw)

    (
        graph,
        atom_types,
        atom_ranks,
        bonds,
        bond_types
    ) = get_graph_from_smiles(smiles)

    X_0 = np.asarray(
        atom_types, dtype=np.int32
    )

    A_0 = np.column_stack([
        np.asarray(
            bonds, dtype=np.int32
        ).reshape(-1, 2),
        np.asarray(
            bond_types, dtype=np.int32
        )
    ])

    n_steps = len(bonds) + 1

    if n_steps > MAX_ITER_PER_MOL:
        raise ValueError(
            f"Molecule needs {n_steps} graph steps"
        )

    fields = [[] for _ in range(10)]
    log_probabilities = []

    for path_idx in range(k):

        step_ids, log_p = traverse_graph(
            graph, atom_ranks, p=p
        )

        X_reordered, A_reordered = single_reorder(
            X_0, A_0, step_ids
        )

        (
            init_i,
            X_i,
            NX_i,
            A_i,
            NA_i,
            actions_i,
            mask_i
        ) = single_expand(
            X_reordered, A_reordered
        )

        path_fields = [
            X_i,
            A_i,
            NX_i,
            NA_i,
            np.zeros_like(NX_i, dtype=np.int32),
            np.full_like(
                NX_i, path_idx, dtype=np.int32
            ),
            np.full_like(
                NX_i, path_idx, dtype=np.int32
            ),
            init_i,
            actions_i,
            mask_i
        ]

        for j, value in enumerate(path_fields):
            fields[j].append(value)

        log_probabilities.append(log_p)

    combined = [
        np.concatenate(items, axis=0)
        for items in fields
    ]

    return (
        *combined,
        np.asarray(
            log_probabilities, dtype=np.float32
        ),
        np.asarray(
            [toxicity_label], dtype=np.int32
        )
    )


# ============================================================
# 5. CONSTRUCT A COMPLETE PYTORCH BATCH
# ============================================================

def collate_graph_records(
    records,
    k=K_PATHS,
    alpha=ALPHA,
    device=DEVICE
):

    B = len(records)

    if B == 0:
        raise ValueError("Cannot construct empty batch")

    fields = [[] for _ in range(12)]
    nsteps_per_molecule = []

    for molecule_idx, record in enumerate(records):

        result = list(
            process_single(record, k=k, p=alpha)
        )

        # Unique IDs across molecules
        result[4] += molecule_idx
        result[6] += molecule_idx * k

        for j, value in enumerate(result):
            fields[j].append(value)

        smiles = record.rsplit(maxsplit=1)[0]
        mol = Chem.MolFromSmiles(smiles)

        nsteps_per_molecule.append(
            mol.GetNumBonds() + 1
        )

    (
        X, A, NX, NA,
        mol_ids, rep_ids, iw_ids,
        action_0, actions,
        last_mask, log_p, tox
    ) = [
        np.concatenate(items, axis=0)
        for items in fields
    ]

    (
        X, A,
        mol_ids_rep, rep_ids_rep,
        iw_ids, last_mask,
        NX, NX_rep,
        action_0, actions, log_p
    ) = merge_single(
        X, A, NX, NA,
        mol_ids, rep_ids, iw_ids,
        action_0, actions,
        last_mask, log_p
    )

    # --------------------------------------------------------
    # RNN sequence indexing
    # --------------------------------------------------------

    graph_to_rnn = np.zeros(
        (B, k, MAX_ITER_PER_MOL),
        dtype=np.int64
    )

    rnn_to_graph = []

    current = 0

    for mol_idx, steps in enumerate(
        nsteps_per_molecule
    ):

        if steps > MAX_ITER_PER_MOL:
            raise ValueError(
                "Molecule exceeds maximum RNN steps"
            )

        for path_idx in range(k):

            graph_to_rnn[
                mol_idx, path_idx, :steps
            ] = np.arange(steps) + current

            for step in range(steps):
                rnn_to_graph.append(
                    (mol_idx, path_idx, step)
                )

            current += steps

    assert current == len(NX)

    rnn_to_graph = np.asarray(
        rnn_to_graph, dtype=np.int64
    )

    # --------------------------------------------------------
    # Graph edges for PyTorch GCN
    # --------------------------------------------------------

    edge_lists = []

    for adjacency in A:

        if len(adjacency) == 0:

            edge_lists.append(
                torch.empty(
                    (2, 0),
                    dtype=torch.long,
                    device=device
                )
            )

        else:

            # Undirected edges: both directions
            both = np.concatenate(
                [
                    adjacency[:, :2],
                    adjacency[:, [1, 0]]
                ],
                axis=0
            )

            edge_lists.append(
                torch.as_tensor(
                    both.T.copy(),
                    dtype=torch.long,
                    device=device
                )
            )

    # --------------------------------------------------------
    # Final model-ready batch
    # --------------------------------------------------------

    batch = {
        "X": torch.as_tensor(
            X.astype(np.int64), device=device
        ),

        "edges": edge_lists,

        "NX": torch.as_tensor(
            NX.astype(np.int64), device=device
        ),

        "NX_rep": torch.as_tensor(
            NX_rep.astype(np.int64), device=device
        ),

        "NX_cum": torch.as_tensor(
            np.cumsum(NX),
            dtype=torch.long,
            device=device
        ),

        "iw_ids": torch.as_tensor(
            iw_ids.astype(np.int64), device=device
        ),

        "last_mask": torch.as_tensor(
            last_mask.astype(np.int64), device=device
        ),

        "action_0": torch.as_tensor(
            action_0.astype(np.int64), device=device
        ),

        "actions": torch.as_tensor(
            actions.astype(np.int64), device=device
        ),

        "log_p_sigma": torch.as_tensor(
            log_p.astype(np.float32), device=device
        ),

        "tox_class": torch.as_tensor(
            tox.astype(np.int64), device=device
        ),

        "graph_to_rnn": torch.as_tensor(
            graph_to_rnn, device=device
        ),

        "rnn_to_graph": torch.as_tensor(
            rnn_to_graph, device=device
        ),

        "batch_size": B,
        "iw_size": k
    }

    # --------------------------------------------------------
    # Tensor consistency checks
    # --------------------------------------------------------

    assert batch["tox_class"].shape == (B,)
    assert batch["action_0"].shape == (B * k,)
    assert batch["log_p_sigma"].shape == (B * k,)
    assert batch["actions"].shape[0] == len(NX)
    assert batch["X"].numel() == int(NX.sum())

    assert len(batch["edges"]) == (
        N_BOND + RECEPTIVE_FIELD
    )

    assert batch["graph_to_rnn"].shape == (
        B, k, MAX_ITER_PER_MOL
    )

    assert batch["rnn_to_graph"].shape == (
        len(NX), 3
    )

    assert torch.isfinite(
        batch["log_p_sigma"]
    ).all()

    return batch


# ============================================================
# 6. TEST WITH REAL AMES-NEGATIVE AND POSITIVE MOLECULES
# ============================================================

print("=" * 65)
print("BLOCK 7: GRAPH BATCH CONSTRUCTION")
print("=" * 65)

# Select actual training molecules from both Ames classes
idx_negative = next(
    int(idx) for idx in train_indices
    if labels[int(idx)] == 0
)

idx_positive = next(
    int(idx) for idx in train_indices
    if labels[int(idx)] == 1
)

test_records = [
    dataset[idx_negative],
    dataset[idx_positive]
]

# Include a ring-containing training molecule to test
# the CONNECT actions in cyclic structures.
ring_idx = None

for idx in train_indices[:2000]:

    idx = int(idx)
    smiles = dataset[idx].rsplit(maxsplit=1)[0]
    mol = Chem.MolFromSmiles(smiles)

    if mol is not None and mol.GetRingInfo().NumRings() > 0:
        ring_idx = idx
        break

assert ring_idx is not None

if ring_idx not in {idx_negative, idx_positive}:
    test_records.append(dataset[ring_idx])

# Construct the batch
batch = collate_graph_records(test_records)

expected_labels = [
    int(record.rsplit(maxsplit=1)[1])
    for record in test_records
]

assert batch["tox_class"].tolist() == expected_labels

print("\n[MOLECULES]")
print("Batch size:", batch["batch_size"])
print("Paths per molecule:", batch["iw_size"])
print("Actual Ames labels:", expected_labels)

print("\n[TENSOR SHAPES]")

for key, value in batch.items():
    if isinstance(value, torch.Tensor):
        print(f"{key:18s}: {tuple(value.shape)}")

print("\nGraph relation types:", len(batch["edges"]))
print("Total graph states:", batch["NX"].numel())
print("Total graph nodes:", batch["X"].numel())

# Action type distribution
action_counts = torch.bincount(
    batch["actions"][:, 0].cpu(),
    minlength=3
)

print("\n[ACTION COUNTS]")
print("Append:", int(action_counts[0]))
print("Connect:", int(action_counts[1]))
print("End:", int(action_counts[2]))

expected_end = len(test_records) * K_PATHS

assert int(action_counts[2]) == expected_end

# Every sampled path has one initialization.
assert batch["action_0"].numel() == expected_end

# Labels must never be duplicated K times.
assert batch["tox_class"].numel() == len(test_records)

# Verify path IDs
assert torch.equal(
    torch.unique(batch["iw_ids"]),
    torch.arange(
        expected_end,
        device=DEVICE
    )
)

# Verify there is at least one ring-closing action
assert int(action_counts[1]) > 0

print("\n[VALIDATION]")
print("Graph batching: PASSED")
print("Action encoding: PASSED")
print("RNN indexing: PASSED")
print("Importance-sampling IDs: PASSED")
print("Toxicity label alignment: PASSED")
print("Ring-closing actions: PASSED")

print("\nBLOCK 7 COMPLETED SUCCESSFULLY!")


BLOCK 7: GRAPH BATCH CONSTRUCTION

[MOLECULES]
Batch size: 2
Paths per molecule: 5
Actual Ames labels: [0, 1]

[TENSOR SHAPES]
X                 : (4977,)
NX                : (320,)
NX_rep            : (4977,)
NX_cum            : (320,)
iw_ids            : (320,)
last_mask         : (4977,)
action_0          : (10,)
actions           : (320, 5)
log_p_sigma       : (10,)
tox_class         : (2,)
graph_to_rnn      : (2, 5, 120)
rnn_to_graph      : (320, 3)

Graph relation types: 6
Total graph states: 320
Total graph nodes: 4977

[ACTION COUNTS]
Append: 275
Connect: 35
End: 10

[VALIDATION]
Graph batching: PASSED
Action encoding: PASSED
RNN indexing: PASSED
Importance-sampling IDs: PASSED
Toxicity label alignment: PASSED
Ring-closing actions: PASSED

BLOCK 7 COMPLETED SUCCESSFULLY!


## GCN Backbone and Atom Embeddings


In [9]:

# ============================================================
# BLOCK 8: CTDDG GRAPH CONVOLUTIONAL BACKBONE
# PyTorch implementation based on the CDGCN architecture
# ============================================================

import torch
from torch import nn
from torch.nn import functional as F

# ============================================================
# 1. SEGMENT SUM
# ============================================================

def sum_segments(x, seg_id, count):
    """
    Sum node features by segment ID.

    Used for graph message aggregation and later
    for intermediate graph representations.
    """

    output = x.new_zeros(
        (count,) + x.shape[1:]
    )

    return output.index_add(
        0, seg_id, x
    )


# ============================================================
# 2. LINEAR LAYER WITH BATCH NORMALIZATION
# ============================================================

class LinearBN(nn.Module):

    def __init__(self, fin, fout):
        super().__init__()

        self.linear = nn.Linear(
            fin,
            fout,
            bias=False
        )

        self.bn = nn.BatchNorm1d(fout)

    def forward(self, x):
        return self.bn(
            self.linear(x)
        )


# ============================================================
# 3. GRAPH CONVOLUTION LAYER
# ============================================================

class GraphConv(nn.Module):
    """
    CDGCN graph convolution.

    Aggregates:
      - Current node features
      - Neighbors for each bond type
      - Distance-2 relationships
      - Distance-3 relationships

    Uses one learnable weight matrix after concatenation.
    """

    def __init__(
        self,
        fin,
        fout,
        n_relations
    ):
        super().__init__()

        self.n_relations = n_relations

        self.W = nn.Parameter(
            torch.empty(
                fin * (1 + n_relations),
                fout
            )
        )

        nn.init.xavier_uniform_(self.W)

    def forward(self, X, edge_lists):

        assert len(edge_lists) == self.n_relations

        # Original node features
        terms = [X]

        # Aggregate messages for each relation
        for edges in edge_lists:

            if edges.numel() == 0:

                aggregated = torch.zeros_like(X)

            else:

                source = edges[1]
                destination = edges[0]

                aggregated = sum_segments(
                    X[source],
                    destination,
                    X.shape[0]
                )

            terms.append(aggregated)

        # Concatenate self and neighbor representations
        combined = torch.cat(
            terms,
            dim=-1
        )

        return combined @ self.W


# ============================================================
# 4. COMPLETE GCN BACKBONE
# ============================================================

class GraphBackbone(nn.Module):

    def __init__(
        self,
        n_atom,
        n_bond,
        D,
        F_e,
        F_h,
        F_skip,
        F_c
    ):
        super().__init__()

        self.n_atom = n_atom
        self.n_bond = n_bond
        self.n_relations = n_bond + D

        # ----------------------------------------------------
        # Atom and last-action-mask embeddings
        # ----------------------------------------------------

        self.atom_embedding = nn.Embedding(
            n_atom,
            F_e
        )

        self.mask_embedding = nn.Embedding(
            3,
            F_e
        )

        # ----------------------------------------------------
        # Graph convolution layers
        # ----------------------------------------------------

        self.convs = nn.ModuleList()
        self.pre_bns = nn.ModuleList()

        for i, fout in enumerate(F_h):

            fin = (
                F_e if i == 0
                else F_h[i - 1]
            )

            if i == 0:
                self.pre_bns.append(nn.Identity())
            else:
                self.pre_bns.append(
                    nn.BatchNorm1d(fin)
                )

            self.convs.append(
                GraphConv(
                    fin,
                    fout,
                    self.n_relations
                )
            )

        # ----------------------------------------------------
        # Skip connection
        # ----------------------------------------------------

        total_hidden = sum(F_h)

        self.bn_skip = nn.BatchNorm1d(
            total_hidden
        )

        self.skip = LinearBN(
            total_hidden,
            F_skip
        )

        # ----------------------------------------------------
        # Final dense layers
        # ----------------------------------------------------

        self.dense = nn.ModuleList()

        dense_inputs = [F_skip] + list(F_c[:-1])

        for fin, fout in zip(
            dense_inputs, F_c
        ):

            self.dense.append(
                LinearBN(fin, fout)
            )

        self.output_dim = F_c[-1]

    def forward(self, batch):

        # ----------------------------------------------------
        # 1. Atom embeddings
        # ----------------------------------------------------

        X = (
            self.atom_embedding(batch["X"])
            + self.mask_embedding(batch["last_mask"])
        )

        # ----------------------------------------------------
        # 2. Graph convolutions
        # ----------------------------------------------------

        layer_outputs = []

        for conv, bn in zip(
            self.convs,
            self.pre_bns
        ):

            if not isinstance(bn, nn.Identity):
                X = F.relu(bn(X))

            X = conv(
                X,
                batch["edges"]
            )

            layer_outputs.append(X)

        # ----------------------------------------------------
        # 3. Skip connection
        # ----------------------------------------------------

        X = torch.cat(
            layer_outputs,
            dim=-1
        )

        X = F.relu(
            self.bn_skip(X)
        )

        X = F.relu(
            self.skip(X)
        )

        # ----------------------------------------------------
        # 4. Dense layers
        # ----------------------------------------------------

        for dense in self.dense:
            X = dense(X)

        return X


# ============================================================
# 5. INITIALIZE AND TEST THE BACKBONE
# ============================================================

print("=" * 65)
print("BLOCK 8: GRAPH CONVOLUTIONAL BACKBONE")
print("=" * 65)

# Rebuild the small test batch on the configured device.
# The previous CPU diagnostic used a CPU batch.
test_batch8 = collate_graph_records(
    test_records,
    k=K_PATHS,
    alpha=ALPHA,
    device=DEVICE
)

# Check the actual GPU device, including its index
actual_device = test_batch8["X"].device

expected_device = (
    torch.device(f"cuda:{torch.cuda.current_device()}")
    if DEVICE.type == "cuda" and DEVICE.index is None
    else DEVICE
)

print("Configured device:", DEVICE)
print("Actual tensor device:", actual_device)
print("Expected tensor device:", expected_device)

assert actual_device == expected_device, (
    f"Device mismatch: expected {expected_device}, "
    f"got {actual_device}"
)

print("Device verification: PASSED")

# Create network
backbone = GraphBackbone(
    n_atom=N_ATOM,
    n_bond=N_BOND,
    D=RECEPTIVE_FIELD,
    F_e=F_E,
    F_h=F_H,
    F_skip=F_SKIP,
    F_c=F_C
).to(DEVICE)

# The reference full CTDDG model applies Xavier
# initialization to linear layers and embeddings.
# We will apply that consistently when constructing
# the complete model in Block 9.

backbone.train()

# ============================================================
# 6. FORWARD PASS TEST
# ============================================================

print("\n[FORWARD PASS]")

X_out = backbone(test_batch8)

expected_shape = (
    test_batch8["X"].numel(),
    F_C[-1]
)

print("Input graph nodes:", test_batch8["X"].numel())
print("Output shape:", tuple(X_out.shape))
print("Expected shape:", expected_shape)

assert tuple(X_out.shape) == expected_shape

assert torch.isfinite(X_out).all(), (
    "Non-finite GCN output detected"
)

print("Forward pass: PASSED")


# ============================================================
# 7. BACKWARD PASS TEST
# ============================================================

print("\n[BACKWARD PASS]")

backbone.zero_grad(set_to_none=True)

# Diagnostic loss only, NOT the toxicity objective.
# Use a subset so BatchNorm does not make the
# squared-feature diagnostic nearly constant.
diagnostic_loss = (
    X_out[:128, :16].square().mean()
)

diagnostic_loss.backward()

assert torch.isfinite(diagnostic_loss)

# Check gradients
grad_count = 0

for name, parameter in backbone.named_parameters():

    if parameter.grad is not None:

        assert torch.isfinite(
            parameter.grad
        ).all(), (
            f"Invalid gradient in {name}"
        )

        grad_count += 1

assert grad_count > 0

assert backbone.atom_embedding.weight.grad is not None
assert backbone.convs[0].W.grad is not None

print("Diagnostic loss:", diagnostic_loss.item())
print("Parameters with gradients:", grad_count)
print("Backward pass: PASSED")


# ============================================================
# 8. MODEL STATISTICS
# ============================================================

total_params = sum(
    p.numel()
    for p in backbone.parameters()
)

trainable_params = sum(
    p.numel()
    for p in backbone.parameters()
    if p.requires_grad
)

print("\n[MODEL CONFIGURATION]")
print("Atom vocabulary:", N_ATOM)
print("Bond vocabulary:", N_BOND)
print("Distance relations:", RECEPTIVE_FIELD)
print("Total graph relations:", N_BOND + RECEPTIVE_FIELD)
print("GCN layers:", len(backbone.convs))
print("Hidden dimensions:", F_H)
print("Output dimension:", F_C[-1])

print("\n[PARAMETERS]")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

print("\n[DEVICE]")
print("Model device:", next(backbone.parameters()).device)
print("Output device:", X_out.device)

if DEVICE.type == "cuda":
    print(
        "GPU peak memory:",
        f"{torch.cuda.max_memory_allocated(DEVICE) / 1024**3:.2f} GB"
    )

print("\n" + "=" * 65)
print("GCN VALIDATION SUMMARY")
print("=" * 65)

print("Atom embeddings: PASSED")
print("Graph convolution layers: PASSED")
print("Skip connections: PASSED")
print("Output dimensions: PASSED")
print("Forward propagation: PASSED")
print("Backward propagation: PASSED")
print("Finite gradients: PASSED")

print("\nBLOCK 8 COMPLETED SUCCESSFULLY!")


BLOCK 8: GRAPH CONVOLUTIONAL BACKBONE
Configured device: cuda
Actual tensor device: cuda:0
Expected tensor device: cuda:0
Device verification: PASSED

[FORWARD PASS]
Input graph nodes: 4951
Output shape: (4951, 512)
Expected shape: (4951, 512)
Forward pass: PASSED

[BACKWARD PASS]
Diagnostic loss: 0.39222604036331177
Parameters with gradients: 26
Backward pass: PASSED

[MODEL CONFIGURATION]
Atom vocabulary: 55
Bond vocabulary: 4
Distance relations: 2
Total graph relations: 6
GCN layers: 6
Hidden dimensions: [32, 64, 128, 128, 256, 256]
Output dimension: 512

[PARAMETERS]
Total parameters: 1,235,744
Trainable parameters: 1,235,744

[DEVICE]
Model device: cuda:0
Output device: cuda:0
GPU peak memory: 0.29 GB

GCN VALIDATION SUMMARY
Atom embeddings: PASSED
Graph convolution layers: PASSED
Skip connections: PASSED
Output dimensions: PASSED
Forward propagation: PASSED
Backward propagation: PASSED
Finite gradients: PASSED

BLOCK 8 COMPLETED SUCCESSFULLY!


## GRU and Molecular Generation Policy

In [10]:

# ============================================================
# BLOCK 9: GRU + MOLECULAR ACTION POLICY
# CTDDG TOXICITY-AWARE PRETRAINING (PYTORCH)
# ============================================================

import torch
from torch import nn
from torch.nn import functional as F

# ============================================================
# 1. MOLECULAR ACTION POLICY
# ============================================================

class Policy(nn.Module):
    """
    Predicts the next molecular generation action.

    APPEND: add an atom with a chosen bond
    CONNECT: connect existing atoms with a bond
    END: terminate molecular generation

    All actions for one graph state are normalized
    together to sum to 1.
    """

    def __init__(self, fin, hidden, n_atom, n_bond):
        super().__init__()

        self.n_atom = n_atom
        self.n_bond = n_bond

        self.linear_h = LinearBN(fin * 2, hidden)
        self.linear_h_t = LinearBN(fin, hidden)

        self.linear_x = nn.Linear(
            hidden,
            n_bond * (n_atom + 1)
        )

        self.linear_x_t = nn.Linear(
            hidden, 1
        )

    def forward(self, X, batch, X_end):

        nx_rep = batch["NX_rep"]
        n_steps = batch["NX"].numel()

        # Combine each node representation with
        # the corresponding GRU graph-state representation.
        node_state = torch.cat(
            [X, X_end[nx_rep]],
            dim=-1
        )

        node_logits = self.linear_x(
            F.relu(self.linear_h(node_state))
        )

        end_logits = self.linear_x_t(
            F.relu(self.linear_h_t(X_end))
        )

        # ----------------------------------------------------
        # Stable joint softmax across all node actions + END
        # ----------------------------------------------------

        node_max = node_logits.max(dim=-1).values

        group_max = end_logits[:, 0].clone()

        group_max = group_max.scatter_reduce(
            0,
            nx_rep,
            node_max,
            reduce="amax",
            include_self=True
        )

        node_weights = torch.exp(
            node_logits - group_max[nx_rep, None]
        )

        end_weights = torch.exp(
            end_logits - group_max[:, None]
        )

        denominator = (
            sum_segments(
                node_weights.sum(
                    dim=-1, keepdim=True
                ),
                nx_rep,
                n_steps
            )
            + end_weights
        )

        soft_node = (
            node_weights / denominator[nx_rep]
        )

        soft_end = (
            end_weights / denominator
        )[:, 0]

        # First N_BOND options are CONNECT actions.
        connect = soft_node[:, :self.n_bond]

        # Remaining options are APPEND actions.
        append = soft_node[:, self.n_bond:].reshape(
            -1,
            self.n_atom,
            self.n_bond
        )

        return append, connect, soft_end


# ============================================================
# 2. COMPLETE CTDDG PRETRAINING NETWORK
# ============================================================

class CTDDGPretrainer(nn.Module):
    """
    CTDDG generative network.

    Components:
      - GCN backbone
      - 3-layer GRU
      - Molecular action policy
      - Initial atom distribution

    Block 10 will implement per-molecule likelihood
    and toxicity-aware Equation 8 using this network.
    """

    def __init__(
        self,
        n_atom,
        n_bond,
        D=2,
        F_e=16,
        F_h=(32, 64, 128, 128, 256, 256),
        F_skip=256,
        F_c=(512,),
        F_policy=128,
        n_rnn=3
    ):
        super().__init__()

        self.n_atom = n_atom
        self.n_bond = n_bond

        # GCN backbone from Block 8
        self.backbone = GraphBackbone(
            n_atom=n_atom,
            n_bond=n_bond,
            D=D,
            F_e=F_e,
            F_h=list(F_h),
            F_skip=F_skip,
            F_c=list(F_c)
        )

        # Sequential graph generation history
        self.rnn = nn.GRU(
            input_size=2 * F_c[-1],
            hidden_size=F_c[-1],
            num_layers=n_rnn,
            batch_first=True
        )

        # Next-action probability distribution
        self.policy = Policy(
            fin=F_c[-1],
            hidden=F_policy,
            n_atom=n_atom,
            n_bond=n_bond
        )

        # Learnable initial atom probabilities
        self.init_atom_logits = nn.Parameter(
            torch.zeros(n_atom)
        )

        # ----------------------------------------------------
        # Xavier initialization from PyTorch reference
        # ----------------------------------------------------

        for module in self.modules():

            if isinstance(
                module, (nn.Linear, nn.Embedding)
            ):
                nn.init.xavier_uniform_(
                    module.weight
                )

                if (
                    getattr(module, "bias", None)
                    is not None
                ):
                    nn.init.zeros_(module.bias)

            elif isinstance(module, nn.GRU):

                for name, parameter in (
                    module.named_parameters()
                ):

                    if "weight" in name:
                        nn.init.xavier_uniform_(parameter)

                    elif "bias" in name:
                        nn.init.zeros_(parameter)

            elif isinstance(module, nn.BatchNorm1d):

                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

    # ========================================================
    # 3. FORWARD PROPAGATION
    # ========================================================

    def forward(self, batch):

        B = batch["batch_size"]
        K = batch["iw_size"]

        # ----------------------------------------------------
        # Step A: Graph convolution
        # ----------------------------------------------------

        X = self.backbone(batch)

        n_steps = batch["NX"].numel()

        # Mean representation of each intermediate graph
        X_avg = sum_segments(
            X,
            batch["NX_rep"],
            n_steps
        ) / batch["NX"][:, None].to(X.dtype)

        # Representation of final node in each graph state
        X_last = X[
            batch["NX_cum"] - 1
        ]

        # Concatenate mean and final-node representations
        features = torch.cat(
            [X_avg, X_last],
            dim=-1
        )

        # ----------------------------------------------------
        # Step B: Prepare graph sequences for GRU
        # ----------------------------------------------------

        # Shape: [B*K, MAX_ITER_PER_MOL, 1024]
        sequences = features[
            batch["graph_to_rnn"]
        ].reshape(
            B * K,
            -1,
            features.shape[-1]
        )

        # Each sampled traversal is one GRU sequence.
        rnn_output, _ = self.rnn(sequences)

        rnn_output = rnn_output.reshape(
            B,
            K,
            -1,
            rnn_output.shape[-1]
        )

        # Recover outputs for actual graph states.
        # Padded steps are not used in the action policy.
        indices = batch["rnn_to_graph"]

        X_end = rnn_output[
            indices[:, 0],
            indices[:, 1],
            indices[:, 2]
        ]

        assert X_end.shape[0] == n_steps

        # ----------------------------------------------------
        # Step C: Molecular action probabilities
        # ----------------------------------------------------

        append, connect, end = self.policy(
            X,
            batch,
            X_end
        )

        return append, connect, end


# ============================================================
# 4. INITIALIZE COMPLETE MODEL
# ============================================================

print("=" * 65)
print("BLOCK 9: GRU + MOLECULAR ACTION POLICY")
print("=" * 65)

# Fix initialization seed for reproducibility.
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

model = CTDDGPretrainer(
    n_atom=N_ATOM,
    n_bond=N_BOND,
    D=RECEPTIVE_FIELD,
    F_e=F_E,
    F_h=F_H,
    F_skip=F_SKIP,
    F_c=F_C,
    F_policy=F_POLICY,
    n_rnn=N_RNN
).to(DEVICE)

model.eval()

# Use verified GPU batch from Block 8.
test_batch9 = test_batch8

model_device = next(model.parameters()).device

assert test_batch9["X"].device == model_device, (
    "Batch and model are on different devices"
)

# ============================================================
# 5. FORWARD PASS TEST
# ============================================================

print("\n[FORWARD PASS]")

with torch.no_grad():
    append, connect, end = model(test_batch9)

n_nodes = test_batch9["X"].numel()
n_steps = test_batch9["NX"].numel()

print("Graph nodes:", n_nodes)
print("Graph states:", n_steps)

print("APPEND shape:", tuple(append.shape))
print("CONNECT shape:", tuple(connect.shape))
print("END shape:", tuple(end.shape))

assert append.shape == (
    n_nodes, N_ATOM, N_BOND
)

assert connect.shape == (
    n_nodes, N_BOND
)

assert end.shape == (n_steps,)

for name, tensor in [
    ("append", append),
    ("connect", connect),
    ("end", end)
]:
    assert torch.isfinite(tensor).all(), (
        f"Non-finite {name} probabilities"
    )

    assert (tensor >= 0).all(), (
        f"Negative {name} probabilities"
    )

    assert (tensor <= 1).all(), (
        f"Invalid {name} probabilities"
    )

print("Output shapes: PASSED")
print("Finite probabilities: PASSED")


# ============================================================
# 6. ACTION PROBABILITY NORMALIZATION
# ============================================================

print("\n[PROBABILITY NORMALIZATION]")

# For every graph state, probabilities across
# every APPEND, CONNECT and END action must sum to 1.

node_probability_sum = (
    append.sum(dim=(1, 2))
    + connect.sum(dim=1)
)

graph_probability_sum = (
    sum_segments(
        node_probability_sum[:, None],
        test_batch9["NX_rep"],
        n_steps
    )[:, 0]
    + end
)

expected = torch.ones_like(
    graph_probability_sum
)

max_error = (
    graph_probability_sum - expected
).abs().max().item()

print("Maximum normalization error:", max_error)

assert torch.allclose(
    graph_probability_sum,
    expected,
    atol=2e-5,
    rtol=1e-5
), "Action probabilities are not normalized!"

print("Joint action probabilities: PASSED")


# ============================================================
# 7. INITIAL ATOM DISTRIBUTION
# ============================================================

print("\n[INITIAL ATOM DISTRIBUTION]")

with torch.no_grad():

    init_probabilities = F.softmax(
        model.init_atom_logits,
        dim=0
    )

assert init_probabilities.shape == (N_ATOM,)

assert torch.allclose(
    init_probabilities.sum(),
    torch.tensor(
        1.0,
        device=model_device
    ),
    atol=1e-6
)

print(
    "Initial atom probability sum:",
    init_probabilities.sum().item()
)

print("Initial atom distribution: PASSED")


# ============================================================
# 8. PARAMETER COUNT
# ============================================================

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\n[MODEL SUMMARY]")
print("Atom types:", N_ATOM)
print("Bond types:", N_BOND)
print("GCN layers:", len(model.backbone.convs))
print("GRU layers:", model.rnn.num_layers)
print("GRU hidden dimension:", model.rnn.hidden_size)
print("Policy hidden dimension:", F_POLICY)

print("\nTotal parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")

print("Device:", model_device)

if model_device.type == "cuda":
    print(
        "GPU peak memory:",
        f"{torch.cuda.max_memory_allocated(model_device) / 1024**3:.2f} GB"
    )

print("\n" + "=" * 65)
print("BLOCK 9 VALIDATION SUMMARY")
print("=" * 65)

print("GCN + GRU integration: PASSED")
print("Molecular action policy: PASSED")
print("Action probability shapes: PASSED")
print("Action normalization: PASSED")
print("Initial atom distribution: PASSED")

print("\nBLOCK 9 COMPLETED SUCCESSFULLY!")


BLOCK 9: GRU + MOLECULAR ACTION POLICY

[FORWARD PASS]
Graph nodes: 4951
Graph states: 320
APPEND shape: (4951, 55, 4)
CONNECT shape: (4951, 4)
END shape: (320,)
Output shapes: PASSED
Finite probabilities: PASSED

[PROBABILITY NORMALIZATION]
Maximum normalization error: 3.5762786865234375e-07
Joint action probabilities: PASSED

[INITIAL ATOM DISTRIBUTION]
Initial atom probability sum: 0.9999999403953552
Initial atom distribution: PASSED

[MODEL SUMMARY]
Atom types: 55
Bond types: 4
GCN layers: 6
GRU layers: 3
GRU hidden dimension: 512
Policy hidden dimension: 128

Total parameters: 6,976,184
Trainable parameters: 6,976,184
Device: cuda:0
GPU peak memory: 0.29 GB

BLOCK 9 VALIDATION SUMMARY
GCN + GRU integration: PASSED
Molecular action policy: PASSED
Action probability shapes: PASSED
Action normalization: PASSED
Initial atom distribution: PASSED

BLOCK 9 COMPLETED SUCCESSFULLY!


## Molecular Log-Likelihood + Toxicity-Aware Loss (Equation 8)

In [11]:

# ============================================================
# BLOCK 10: CTDDG MOLECULAR LIKELIHOOD + TOXICITY LOSS
# Based on the PyTorch CTDDG Equation 8 reference
# ============================================================

import math
import torch
from torch.nn import functional as F

# ============================================================
# 1. IMPORTANCE-WEIGHTED MOLECULAR LOG-LIKELIHOOD
# ============================================================

def molecular_log_likelihood(model, batch):
    """
    Compute one importance-weighted log-likelihood
    estimate per molecule.

    Each molecule has K graph traversal paths.

    Returns:
        ll: tensor of shape [B]
    """

    B = batch["batch_size"]
    K = batch["iw_size"]

    # --------------------------------------------------------
    # A. Get action probabilities from GCN + GRU + policy
    # --------------------------------------------------------

    append, connect, end = model(batch)

    actions = batch["actions"]
    action_type = actions[:, 0]

    # --------------------------------------------------------
    # B. Initial atom log-probability
    # --------------------------------------------------------

    init_log_probs = F.log_softmax(
        model.init_atom_logits,
        dim=0
    )

    init_ll = init_log_probs[
        batch["action_0"]
    ]

    assert init_ll.shape == (B * K,)

    # --------------------------------------------------------
    # C. Action log-probabilities
    # --------------------------------------------------------

    eps = 1e-10

    # APPEND:
    # actions[:,3] = atom position
    # actions[:,1] = new atom type
    # actions[:,2] = bond type

    append_ll = torch.log(
        append[
            actions[:, 3],
            actions[:, 1],
            actions[:, 2]
        ] + eps
    )

    # CONNECT:
    # actions[:,4] = atom position
    # actions[:,2] = bond type

    connect_ll = torch.log(
        connect[
            actions[:, 4],
            actions[:, 2]
        ] + eps
    )

    # END action probability
    end_ll = torch.log(end + eps)

    # Select the actual action taken at each graph step
    action_ll = torch.where(
        action_type == 0,
        append_ll,
        torch.where(
            action_type == 1,
            connect_ll,
            end_ll
        )
    )

    assert action_ll.shape == (
        batch["NX"].numel(),
    )

    # --------------------------------------------------------
    # D. Sum action likelihoods for each sampled path
    # --------------------------------------------------------

    path_ll = sum_segments(
        action_ll.unsqueeze(1),
        batch["iw_ids"],
        B * K
    )[:, 0]

    # Include the initial atom choice
    path_ll = path_ll + init_ll

    assert path_ll.shape == (B * K,)

    # --------------------------------------------------------
    # E. Importance-sampling correction
    # --------------------------------------------------------

    path_ll = path_ll.reshape(B, K)

    proposal_logp = batch["log_p_sigma"].reshape(
        B, K
    )

    importance_adjusted_ll = (
        path_ll - proposal_logp
    )

    # Log-mean-exp over K sampled graph paths
    molecule_ll = (
        torch.logsumexp(
            importance_adjusted_ll,
            dim=1
        )
        - math.log(K)
    )

    assert molecule_ll.shape == (B,)

    return molecule_ll


# ============================================================
# 2. TOXICITY-AWARE EQUATION 8
# ============================================================

def toxicity_objective(ll, labels, lam=1.0):
    """
    Signed CTDDG toxicity-aware loss.

    Ames-negative (0):
        contribution = -log P(molecule)

    Ames-positive (1):
        contribution = +lambda * log P(molecule)

    Lambda controls the toxicity penalty.
    """

    if ll.ndim != 1:
        raise ValueError(
            "Log-likelihood must have shape [B]"
        )

    if labels.shape != ll.shape:
        raise ValueError(
            "Ames labels and likelihoods must align"
        )

    if lam < 0:
        raise ValueError(
            "Lambda must be nonnegative"
        )

    if not torch.all(
        (labels == 0) | (labels == 1)
    ):
        raise ValueError(
            "Ames labels must be binary (0 or 1)"
        )

    weights = (
        1.0
        - (1.0 + lam) * labels.to(ll.dtype)
    )

    loss = -(weights * ll).mean()

    return loss


# ============================================================
# 3. MATHEMATICAL SIGN AND GRADIENT TEST
# ============================================================

print("=" * 65)
print("BLOCK 10: MOLECULAR LIKELIHOOD + TOXICITY LOSS")
print("=" * 65)

print("\n[TEST 1: EQUATION 8 MATHEMATICS]")

toy_ll = torch.tensor(
    [-2.0, -3.0],
    dtype=torch.float32,
    requires_grad=True
)

toy_labels = torch.tensor(
    [0, 1],
    dtype=torch.long
)

toy_loss = toxicity_objective(
    toy_ll,
    toy_labels,
    lam=1.0
)

toy_loss.backward()

expected_loss = torch.tensor(-0.5)

expected_grads = torch.tensor(
    [-0.5, 0.5]
)

assert torch.allclose(
    toy_loss.detach(),
    expected_loss
)

assert torch.allclose(
    toy_ll.grad,
    expected_grads
)

print("Ames-negative likelihood:", -2.0)
print("Ames-positive likelihood:", -3.0)
print("Lambda:", 1.0)
print("Computed loss:", toy_loss.item())
print("Expected loss:", expected_loss.item())
print("Gradients:", toy_ll.grad.tolist())

print("Equation 8 sign test: PASSED")
print("Equation 8 gradient test: PASSED")


# ============================================================
# 4. VERIFY LAMBDA BEHAVIOUR
# ============================================================

print("\n[TEST 2: LAMBDA BEHAVIOUR]")

# Test that different lambda values produce the
# mathematically expected objective.

for lam in [0.0, 0.25, 0.5, 1.0, 2.0]:

    ll_test = torch.tensor(
        [-2.0, -3.0]
    )

    labels_test = torch.tensor(
        [0, 1]
    )

    loss_value = toxicity_objective(
        ll_test,
        labels_test,
        lam=lam
    )

    expected_value = (
        2.0 - 3.0 * lam
    ) / 2.0

    assert torch.isclose(
        loss_value,
        torch.tensor(expected_value),
        atol=1e-6
    )

    print(
        f"Lambda = {lam:4.2f} | "
        f"Loss = {loss_value.item():+.6f}"
    )

print("Lambda behaviour: PASSED")


# ============================================================
# 5. VERIFY REAL MOLECULAR LOG-LIKELIHOOD
# ============================================================

print("\n[TEST 3: REAL CHEMBL MOLECULE LIKELIHOOD]")

# Use the verified model and batch from Block 9.
model.eval()

with torch.no_grad():

    ll = molecular_log_likelihood(
        model,
        test_batch9
    )

    actual_labels = test_batch9[
        "tox_class"
    ]

    assert ll.shape == (
        test_batch9["batch_size"],
    )

    assert torch.isfinite(ll).all(), (
        "Non-finite molecular log-likelihood"
    )

    assert actual_labels.shape == ll.shape

    real_loss = toxicity_objective(
        ll,
        actual_labels,
        lam=TOX_LAMBDA
    )

    assert torch.isfinite(real_loss)

    print("Batch size:", len(ll))
    print("Paths per molecule:", K_PATHS)

    for i in range(len(ll)):

        print(
            f"Molecule {i + 1} | "
            f"Ames = {actual_labels[i].item()} | "
            f"Log-likelihood = {ll[i].item():.6f}"
        )

    print(
        "\nToxicity-aware loss:",
        real_loss.item()
    )

print("Real molecule likelihood: PASSED")
print("Real Ames label alignment: PASSED")


# ============================================================
# 6. TEST END-TO-END GRADIENT FLOW
# ============================================================

print("\n[TEST 4: END-TO-END BACKPROPAGATION]")

model.train()
model.zero_grad(set_to_none=True)

ll_train = molecular_log_likelihood(
    model,
    test_batch9
)

loss_train = toxicity_objective(
    ll_train,
    test_batch9["tox_class"],
    lam=TOX_LAMBDA
)

assert torch.isfinite(loss_train)

loss_train.backward()

# Check that gradients reach every major component
important_parameters = {
    "Atom embeddings":
        model.backbone.atom_embedding.weight,

    "First GCN layer":
        model.backbone.convs[0].W,

    "GRU input weights":
        model.rnn.weight_ih_l0,

    "Policy output":
        model.policy.linear_x.weight,

    "Initial atom distribution":
        model.init_atom_logits
}

for name, parameter in important_parameters.items():

    grad = parameter.grad

    assert grad is not None, (
        f"No gradient for {name}"
    )

    assert torch.isfinite(grad).all(), (
        f"Non-finite gradient for {name}"
    )

    grad_norm = grad.norm().item()

    assert grad_norm > 0, (
        f"Zero gradient for {name}"
    )

    print(
        f"{name:28s} | "
        f"Gradient norm: {grad_norm:.6f}"
    )

# Count all trainable parameters receiving gradients
gradient_parameter_count = sum(
    p.numel()
    for p in model.parameters()
    if p.grad is not None
)

total_parameter_count = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    "\nParameters receiving gradients:",
    f"{gradient_parameter_count:,}"
)

print(
    "Total trainable parameters:",
    f"{total_parameter_count:,}"
)

# No optimizer step yet: weights remain unchanged.
model.zero_grad(set_to_none=True)
model.eval()

print("\nEnd-to-end gradient flow: PASSED")


# ============================================================
# 7. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 65)
print("BLOCK 10 VALIDATION SUMMARY")
print("=" * 65)

print("Importance-weighted likelihood: PASSED")
print("Toxicity objective mathematics: PASSED")
print("Lambda handling: PASSED")
print("Real molecule labels: PASSED")
print("GCN + GRU + policy gradients: PASSED")

print("\nCurrent toxicity lambda:", TOX_LAMBDA)
print("Model weights updated: NO")

print("\nBLOCK 10 COMPLETED SUCCESSFULLY!")


BLOCK 10: MOLECULAR LIKELIHOOD + TOXICITY LOSS

[TEST 1: EQUATION 8 MATHEMATICS]
Ames-negative likelihood: -2.0
Ames-positive likelihood: -3.0
Lambda: 1.0
Computed loss: -0.5
Expected loss: -0.5
Gradients: [-0.5, 0.5]
Equation 8 sign test: PASSED
Equation 8 gradient test: PASSED

[TEST 2: LAMBDA BEHAVIOUR]
Lambda = 0.00 | Loss = +1.000000
Lambda = 0.25 | Loss = +0.625000
Lambda = 0.50 | Loss = +0.250000
Lambda = 1.00 | Loss = -0.500000
Lambda = 2.00 | Loss = -2.000000
Lambda behaviour: PASSED

[TEST 3: REAL CHEMBL MOLECULE LIKELIHOOD]
Batch size: 2
Paths per molecule: 5
Molecule 1 | Ames = 0 | Log-likelihood = -244.737701
Molecule 2 | Ames = 1 | Log-likelihood = -348.407135

Toxicity-aware loss: -51.834716796875
Real molecule likelihood: PASSED
Real Ames label alignment: PASSED

[TEST 4: END-TO-END BACKPROPAGATION]
Atom embeddings              | Gradient norm: 13.255037
First GCN layer              | Gradient norm: 49.586296
GRU input weights            | Gradient norm: 36.458229
Polic

## Model Integrity and Unit Tests

In [12]:

# ============================================================
# BLOCK 11: CTDDG MODEL INTEGRITY AND UNIT TESTS
# ============================================================

import math
import torch
import numpy as np
from rdkit import Chem
from torch.nn import functional as F

print("=" * 65)
print("BLOCK 11: MODEL INTEGRITY AND UNIT TESTS")
print("=" * 65)

# Use the already verified batch and model.
b = test_batch9
B = b["batch_size"]
K = b["iw_size"]
n_steps = b["NX"].numel()
n_nodes = b["X"].numel()

device = next(model.parameters()).device

assert b["X"].device == device

# ============================================================
# TEST 1: GRAPH CONSTRUCTION ACTIONS
# ============================================================

print("\n[TEST 1: GRAPH ACTION CONSISTENCY]")

paths_checked = 0

for record in test_records:

    smiles = record.rsplit(maxsplit=1)[0]

    sampled_paths = sample_graph_paths(
        smiles,
        k=3,
        alpha=ALPHA
    )

    for path in sampled_paths:

        X_0 = path["X"]
        A_0 = path["A"]

        (
            init_atom,
            X,
            NX,
            A,
            NA,
            actions,
            last_mask
        ) = single_expand(X_0, A_0)

        # One initialization and one END per path.
        assert int(init_atom[0]) == int(X_0[0])
        assert len(actions) == len(A_0) + 1
        assert actions[-1, 0] == 2

        current_atoms = 1

        for step, bond in enumerate(A_0):

            action = actions[step]

            begin = int(bond[0])
            end = int(bond[1])
            bond_type = int(bond[2])

            assert int(action[2]) == bond_type

            if begin < end:

                # APPEND action introduces one new atom.
                assert int(action[0]) == 0
                assert end == current_atoms
                assert 0 <= begin < current_atoms

                assert int(action[1]) == int(X_0[end])
                assert int(action[3]) == begin

                current_atoms += 1

            else:

                # CONNECT action uses existing atoms.
                assert int(action[0]) == 1
                assert 0 <= begin < current_atoms
                assert 0 <= end < current_atoms
                assert int(action[4]) == end

        assert current_atoms == len(X_0)
        assert NX[-1] == len(X_0)

        # Each intermediate graph contains
        # all previously processed bonds.
        assert np.array_equal(
            NA,
            np.arange(len(A_0) + 1)
        )

        paths_checked += 1

print("Graph paths verified:", paths_checked)
print("Append/connect/end consistency: PASSED")


# ============================================================
# TEST 2: BATCH GRAPH AND RNN INDEXING
# ============================================================

print("\n[TEST 2: BATCH INDEX INTEGRITY]")

# Every atom belongs to exactly one graph state.
assert b["NX_rep"].numel() == n_nodes

assert torch.equal(
    torch.bincount(
        b["NX_rep"],
        minlength=n_steps
    ),
    b["NX"]
)

# All graph edges must remain within a graph state.
for relation_idx, edges in enumerate(b["edges"]):

    assert edges.shape[0] == 2

    if edges.numel() == 0:
        continue

    assert int(edges.min()) >= 0
    assert int(edges.max()) < n_nodes

    source_graph = b["NX_rep"][edges[0]]
    target_graph = b["NX_rep"][edges[1]]

    assert torch.equal(
        source_graph,
        target_graph
    ), (
        f"Cross-graph edge detected "
        f"in relation {relation_idx}"
    )

# Check every actual RNN state index.
rnn_to_graph = b["rnn_to_graph"].cpu().numpy()
graph_to_rnn = b["graph_to_rnn"].cpu().numpy()

for flat_idx, (mol_id, path_id, step) in enumerate(
    rnn_to_graph
):

    assert graph_to_rnn[
        mol_id, path_id, step
    ] == flat_idx

# Verify that each path has the correct number of steps.
for mol_id, record in enumerate(test_records):

    smiles = record.rsplit(maxsplit=1)[0]
    mol = Chem.MolFromSmiles(smiles)

    expected_steps = mol.GetNumBonds() + 1

    for path_id in range(K):

        actual_steps = int(
            np.sum(
                (rnn_to_graph[:, 0] == mol_id)
                & (rnn_to_graph[:, 1] == path_id)
            )
        )

        assert actual_steps == expected_steps

# Verify labels are attached to molecules, not paths.
expected_labels = [
    int(record.rsplit(maxsplit=1)[1])
    for record in test_records
]

assert b["tox_class"].tolist() == expected_labels
assert b["tox_class"].numel() == B

# Verify importance-sampling path IDs.
expected_path_ids = torch.arange(
    B * K,
    device=device
)

assert torch.equal(
    torch.unique(b["iw_ids"]),
    expected_path_ids
)

# Every path ends with one END action.
action_counts = torch.bincount(
    b["actions"][:, 0],
    minlength=3
)

assert int(action_counts[2]) == B * K

print("Graph states:", n_steps)
print("Graph nodes:", n_nodes)
print("Graph relation types:", len(b["edges"]))
print("RNN state mapping: PASSED")
print("No cross-graph edges: PASSED")
print("Ames label alignment: PASSED")
print("Importance path indexing: PASSED")


# ============================================================
# TEST 3: INDEPENDENT LOG-LIKELIHOOD CALCULATION
# ============================================================

print("\n[TEST 3: INDEPENDENT LIKELIHOOD VERIFICATION]")

# Recalculate the molecular likelihood by explicitly
# iterating over each graph action.
#
# This independently checks the vectorized likelihood
# implementation introduced in Block 10.

model.eval()

with torch.no_grad():

    append, connect, end = model(b)

    vectorized_ll = molecular_log_likelihood(
        model, b
    )

    init_log_probs = F.log_softmax(
        model.init_atom_logits,
        dim=0
    )

    manual_ll = []

    for mol_id in range(B):

        path_scores = []

        for path_id in range(K):

            flat_path_id = mol_id * K + path_id

            graph_indices = np.flatnonzero(
                (rnn_to_graph[:, 0] == mol_id)
                & (rnn_to_graph[:, 1] == path_id)
            )

            # Initial atom log-probability.
            init_atom_id = int(
                b["action_0"][flat_path_id].item()
            )

            score = init_log_probs[init_atom_id]

            # Sum log-probabilities of actual actions.
            for graph_idx in graph_indices:

                action = b["actions"][graph_idx]

                action_type = int(action[0].item())

                if action_type == 0:

                    # APPEND
                    atom_type = int(action[1].item())
                    bond_type = int(action[2].item())
                    position = int(action[3].item())

                    probability = append[
                        position,
                        atom_type,
                        bond_type
                    ]

                elif action_type == 1:

                    # CONNECT
                    bond_type = int(action[2].item())
                    position = int(action[4].item())

                    probability = connect[
                        position,
                        bond_type
                    ]

                elif action_type == 2:

                    # END
                    probability = end[graph_idx]

                else:
                    raise AssertionError(
                        "Unknown molecular action type"
                    )

                score = score + torch.log(
                    probability + 1e-10
                )

            # Correct for proposal distribution Q_alpha.
            score = score - b[
                "log_p_sigma"
            ][flat_path_id]

            path_scores.append(score)

        # Importance-weighted log-mean-exp over K paths.
        molecule_score = (
            torch.logsumexp(
                torch.stack(path_scores),
                dim=0
            )
            - math.log(K)
        )

        manual_ll.append(molecule_score)

    manual_ll = torch.stack(manual_ll)

    max_difference = (
        manual_ll - vectorized_ll
    ).abs().max().item()

    print(
        "Vectorized log-likelihoods:",
        vectorized_ll.cpu().tolist()
    )

    print(
        "Manual log-likelihoods:",
        manual_ll.cpu().tolist()
    )

    print(
        "Maximum absolute difference:",
        max_difference
    )

    assert torch.allclose(
        manual_ll,
        vectorized_ll,
        atol=2e-3,
        rtol=1e-5
    ), (
        "Manual and vectorized likelihoods differ"
    )

print("Importance-weighted likelihood: PASSED")


# ============================================================
# TEST 4: GRADIENT CLIPPING WITHOUT WEIGHT UPDATES
# ============================================================

print("\n[TEST 4: GRADIENT CLIPPING]")

model.train()
model.zero_grad(set_to_none=True)

# Snapshot one parameter to verify that this
# test does not update model weights.
weight_before = (
    model.backbone.convs[0].W.detach().clone()
)

ll_train = molecular_log_likelihood(
    model, b
)

loss_train = toxicity_objective(
    ll_train,
    b["tox_class"],
    lam=TOX_LAMBDA
)

assert torch.isfinite(loss_train)

loss_train.backward()

# Validate every available gradient.
for name, parameter in model.named_parameters():

    if parameter.grad is not None:

        assert torch.isfinite(
            parameter.grad
        ).all(), (
            f"Non-finite gradient: {name}"
        )

# Clip global gradient norm to configured threshold.
grad_norm_before = torch.nn.utils.clip_grad_norm_(
    model.parameters(),
    max_norm=GRAD_CLIP,
    error_if_nonfinite=True
)

# Calculate the global norm after clipping.
grad_norm_after = torch.linalg.vector_norm(
    torch.stack([
        p.grad.detach().float().norm()
        for p in model.parameters()
        if p.grad is not None
    ])
)

assert torch.isfinite(grad_norm_after)
assert grad_norm_after.item() <= GRAD_CLIP + 1e-4

# Backward propagation and gradient clipping
# must not change the model weights.
assert torch.equal(
    weight_before,
    model.backbone.convs[0].W.detach()
)

print(
    "Gradient norm before clipping:",
    float(grad_norm_before)
)

print(
    "Gradient norm after clipping:",
    float(grad_norm_after)
)

print("Gradient clipping: PASSED")
print("Model weights unchanged: PASSED")

# Clean up gradients.
model.zero_grad(set_to_none=True)
model.eval()


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 65)
print("BLOCK 11 VALIDATION SUMMARY")
print("=" * 65)

print("Graph action replay: PASSED")
print("Graph state integrity: PASSED")
print("RNN sequence indexing: PASSED")
print("Molecule/label alignment: PASSED")
print("Independent likelihood verification: PASSED")
print("Finite gradients: PASSED")
print("Gradient clipping: PASSED")
print("Model weights updated: NO")

print("\nBLOCK 11 COMPLETED SUCCESSFULLY!")


BLOCK 11: MODEL INTEGRITY AND UNIT TESTS

[TEST 1: GRAPH ACTION CONSISTENCY]
Graph paths verified: 6
Append/connect/end consistency: PASSED

[TEST 2: BATCH INDEX INTEGRITY]
Graph states: 320
Graph nodes: 4951
Graph relation types: 6
RNN state mapping: PASSED
No cross-graph edges: PASSED
Ames label alignment: PASSED
Importance path indexing: PASSED

[TEST 3: INDEPENDENT LIKELIHOOD VERIFICATION]
Vectorized log-likelihoods: [-228.0640106201172, -313.6793518066406]
Manual log-likelihoods: [-228.0640106201172, -313.6794128417969]
Maximum absolute difference: 6.103515625e-05
Importance-weighted likelihood: PASSED

[TEST 4: GRADIENT CLIPPING]
Gradient norm before clipping: 157.74957275390625
Gradient norm after clipping: 3.000000238418579
Gradient clipping: PASSED
Model weights unchanged: PASSED

BLOCK 11 VALIDATION SUMMARY
Graph action replay: PASSED
Graph state integrity: PASSED
RNN sequence indexing: PASSED
Molecule/label alignment: PASSED
Independent likelihood verification: PASSED
Finite

## Training Loop, Validation and Checkpoints

In [13]:

# ============================================================
# BLOCK 12: TRAINING LOOP + VALIDATION + CHECKPOINTS
# CTDDG TOXICITY-AWARE PRETRAINING
# ============================================================

import os
import json
import math
import time
import random
import hashlib
from pathlib import Path

import numpy as np
import torch

# ============================================================
# 1. RUN CONFIGURATION
# ============================================================

RUNS_DIR = PRETRAINING_DIR / "toxicity_aware" / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_CONFIG = {
    "n_atom": N_ATOM,
    "n_bond": N_BOND,
    "D": RECEPTIVE_FIELD,
    "F_e": F_E,
    "F_h": list(F_H),
    "F_skip": F_SKIP,
    "F_c": list(F_C),
    "F_policy": F_POLICY,
    "n_rnn": N_RNN,
}

def hash_indices(indices):
    arr = np.asarray(indices, dtype="<i8")
    return hashlib.sha256(arr.tobytes()).hexdigest()


def make_fresh_model(seed=SEED):
    """
    Create identical initial weights for comparable runs
    without permanently changing the global PyTorch RNG.
    """
    cpu_rng = torch.get_rng_state()

    cuda_rng = (
        torch.cuda.get_rng_state_all()
        if torch.cuda.is_available()
        else None
    )

    try:
        torch.manual_seed(seed)

        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        net = CTDDGPretrainer(
            **MODEL_CONFIG
        ).to(DEVICE)

    finally:
        torch.set_rng_state(cpu_rng)

        if cuda_rng is not None:
            torch.cuda.set_rng_state_all(cuda_rng)

    return net


# ============================================================
# 2. REPRODUCIBLE LENGTH-BUCKETED BATCHING
# ============================================================

def make_epoch_batches(
    sorted_ids,
    epoch,
    batch_size=BATCH_SIZE,
    seed=SEED
):
    """
    Create batches of similarly sized molecules.

    The sorted dataset indices are prepared once per run.
    Shuffling is deterministic for each epoch.
    """

    rng = random.Random(seed + epoch)

    # Shuffle within local length buckets.
    bucket_size = max(batch_size * 32, 256)

    batches = []

    for start in range(0, len(sorted_ids), bucket_size):

        bucket = list(
            sorted_ids[start:start + bucket_size]
        )

        rng.shuffle(bucket)

        for offset in range(0, len(bucket), batch_size):

            selected = bucket[
                offset:offset + batch_size
            ]

            if len(selected) == batch_size:
                batches.append(selected)

    rng.shuffle(batches)

    return batches


# ============================================================
# 3. INITIALIZE OR RESUME A TRAINING RUN
# ============================================================

def create_training_run(
    run_name,
    lam,
    learning_rate,
    train_ids,
    val_ids,
    resume=False
):
    if not run_name or not all(
        c.isalnum() or c in "_-"
        for c in run_name
    ):
        raise ValueError("Invalid run name")

    if lam < 0 or learning_rate <= 0:
        raise ValueError("Invalid hyperparameters")

    train_ids = np.asarray(
        train_ids, dtype=np.int64
    )

    val_ids = np.asarray(
        val_ids, dtype=np.int64
    )

    assert len(train_ids) >= BATCH_SIZE
    assert len(val_ids) > 0

    assert len(np.unique(train_ids)) == len(train_ids)
    assert len(np.unique(val_ids)) == len(val_ids)

    assert len(
        np.intersect1d(train_ids, val_ids)
    ) == 0

    assert np.all(train_ids >= 0)
    assert np.all(val_ids >= 0)

    assert np.all(train_ids < len(dataset))
    assert np.all(val_ids < len(dataset))

    run_dir = RUNS_DIR / run_name

    config_path = run_dir / "config.json"
    checkpoint_path = run_dir / "checkpoint.pt"
    best_path = run_dir / "model_best.pt"
    train_log_path = run_dir / "train_log.jsonl"
    val_log_path = run_dir / "validation_log.jsonl"

    config = {
        "model_config": MODEL_CONFIG,
        "seed": SEED,
        "tox_lambda": float(lam),
        "learning_rate": float(learning_rate),
        "weight_decay": float(WEIGHT_DECAY),
        "batch_size": int(BATCH_SIZE),
        "k_paths": int(K_PATHS),
        "alpha": float(ALPHA),
        "grad_clip": float(GRAD_CLIP),
        "lr_decay": float(LR_DECAY),
        "lr_decay_step": int(LR_DECAY_STEP),
        "train_count": len(train_ids),
        "val_count": len(val_ids),
        "train_indices_sha256": hash_indices(train_ids),
        "val_indices_sha256": hash_indices(val_ids),
        "chembl_sha256": chembl_hash,
        "atom_vocab_sha256": sha256_file(
            ATOM_TYPES_FILE
        ),
    }

    if resume:

        if not (
            config_path.exists()
            and checkpoint_path.exists()
        ):
            raise FileNotFoundError(
                "Resume requested but checkpoint "
                "or configuration is missing"
            )

        saved_config = json.loads(
            config_path.read_text(encoding="utf-8")
        )

        if saved_config != config:
            raise ValueError(
                "Checkpoint configuration mismatch"
            )

    else:

        if run_dir.exists() and any(run_dir.iterdir()):
            raise FileExistsError(
                f"Run already exists: {run_dir}. "
                "Choose another run name or resume."
            )

        run_dir.mkdir(parents=True, exist_ok=True)

        config_path.write_text(
            json.dumps(config, indent=2),
            encoding="utf-8"
        )

    # Sort once by SMILES record length.
    sorted_ids = sorted(
        map(int, train_ids),
        key=lambda idx: len(dataset[idx])
    )

    net = make_fresh_model()

    optimizer = torch.optim.Adam(
        net.parameters(),
        lr=learning_rate,
        weight_decay=WEIGHT_DECAY
    )

    run = {
        "name": run_name,
        "dir": run_dir,
        "config": config,
        "model": net,
        "optimizer": optimizer,
        "train_ids": train_ids,
        "val_ids": val_ids,
        "sorted_ids": sorted_ids,
        "step": 0,
        "epoch": 0,
        "batch_offset": 0,
        "best_val_loss": float("inf"),
        "best_step": None,
        "checkpoint_path": checkpoint_path,
        "best_path": best_path,
        "train_log_path": train_log_path,
        "val_log_path": val_log_path,
    }

    if resume:

        # Load only checkpoints created by our own code.
        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=False
        )

        net.load_state_dict(
            checkpoint["model_state_dict"]
        )

        optimizer.load_state_dict(
            checkpoint["optimizer_state_dict"]
        )

        run["step"] = int(checkpoint["step"])
        run["epoch"] = int(checkpoint["epoch"])
        run["batch_offset"] = int(
            checkpoint["batch_offset"]
        )

        run["best_val_loss"] = float(
            checkpoint["best_val_loss"]
        )

        run["best_step"] = checkpoint["best_step"]

        random.setstate(
            checkpoint["python_rng"]
        )

        np.random.set_state(
            checkpoint["numpy_rng"]
        )

        torch.set_rng_state(
            checkpoint["torch_rng"]
        )

        if (
            torch.cuda.is_available()
            and checkpoint["cuda_rng"] is not None
        ):
            torch.cuda.set_rng_state_all(
                checkpoint["cuda_rng"]
            )

        print(
            f"Resumed {run_name} "
            f"at step {run['step']}"
        )

    else:
        # Same path-sampling seed for comparable runs.
        random.seed(SEED)

    return run


# ============================================================
# 4. SAVE RESUMABLE CHECKPOINTS
# ============================================================

def save_training_checkpoint(run):

    checkpoint = {
        "step": run["step"],
        "epoch": run["epoch"],
        "batch_offset": run["batch_offset"],
        "model_state_dict":
            run["model"].state_dict(),
        "optimizer_state_dict":
            run["optimizer"].state_dict(),
        "best_val_loss": run["best_val_loss"],
        "best_step": run["best_step"],
        "config": run["config"],
        "python_rng": random.getstate(),
        "numpy_rng": np.random.get_state(),
        "torch_rng": torch.get_rng_state(),
        "cuda_rng": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else None
        ),
    }

    path = run["checkpoint_path"]
    temp_path = path.with_suffix(".tmp")

    torch.save(checkpoint, temp_path)
    os.replace(temp_path, path)

    return path


# ============================================================
# 5. VALIDATION ON HELD-OUT MOLECULES
# ============================================================

@torch.no_grad()
def evaluate_training_run(run):

    net = run["model"]

    was_training = net.training
    rng_state = random.getstate()

    net.eval()

    total_loss = 0.0
    total_count = 0

    class_ll_sum = {0: 0.0, 1: 0.0}
    class_count = {0: 0, 1: 0}

    try:
        # Identical graph paths for each validation.
        random.seed(VAL_SEED + 12345)

        ids = run["val_ids"]

        for offset in range(
            0, len(ids), BATCH_SIZE
        ):

            selected_ids = ids[
                offset:offset + BATCH_SIZE
            ]

            records = [
                dataset[int(idx)]
                for idx in selected_ids
            ]

            batch = collate_graph_records(
                records,
                k=K_PATHS,
                alpha=ALPHA,
                device=DEVICE
            )

            ll = molecular_log_likelihood(
                net, batch
            )

            loss = toxicity_objective(
                ll,
                batch["tox_class"],
                lam=run["config"]["tox_lambda"]
            )

            if not torch.isfinite(loss):
                raise FloatingPointError(
                    "Non-finite validation loss"
                )

            n = len(records)

            total_loss += float(loss.item()) * n
            total_count += n

            for cls in (0, 1):

                mask = batch["tox_class"] == cls

                if mask.any():

                    class_ll_sum[cls] += float(
                        ll[mask].sum().item()
                    )

                    class_count[cls] += int(
                        mask.sum().item()
                    )

    finally:
        random.setstate(rng_state)
        net.train(was_training)

    metrics = {
        "step": run["step"],
        "val_loss": total_loss / total_count,
        "val_neg_ll_mean": (
            class_ll_sum[0] / class_count[0]
            if class_count[0] else None
        ),
        "val_pos_ll_mean": (
            class_ll_sum[1] / class_count[1]
            if class_count[1] else None
        ),
        "n_val_molecules": total_count,
        "n_val_negative": class_count[0],
        "n_val_positive": class_count[1],
    }

    with run["val_log_path"].open(
        "a", encoding="utf-8"
    ) as f:
        f.write(json.dumps(metrics) + "\n")

    # "Best" here means lowest signed validation
    # loss for this particular lambda only.
    if metrics["val_loss"] < run["best_val_loss"]:

        run["best_val_loss"] = metrics["val_loss"]
        run["best_step"] = run["step"]

        best_checkpoint = {
            "model_state_dict":
                net.state_dict(),
            "config": run["config"],
            "step": run["step"],
            "validation": metrics,
        }

        temp_path = run["best_path"].with_suffix(
            ".tmp"
        )

        torch.save(best_checkpoint, temp_path)

        os.replace(
            temp_path,
            run["best_path"]
        )

    return metrics


# ============================================================
# 6. TRAINING LOOP
# ============================================================

def train_steps(
    run,
    num_steps,
    validation_every=100,
    checkpoint_every=100,
    log_every=10
):
    """
    Train for a controlled number of optimizer steps.

    No full training starts unless explicitly requested.
    """

    if num_steps <= 0:
        return []

    net = run["model"]
    optimizer = run["optimizer"]

    net.train()

    start_time = time.perf_counter()
    history = []

    completed = 0

    while completed < num_steps:

        epoch = run["epoch"]

        batches = make_epoch_batches(
            run["sorted_ids"],
            epoch=epoch,
            batch_size=BATCH_SIZE,
            seed=SEED
        )

        if not batches:
            raise ValueError("No valid training batches")

        offset = run["batch_offset"]

        if offset >= len(batches):
            raise ValueError(
                "Invalid resume batch offset"
            )

        for batch_idx in range(offset, len(batches)):

            selected_ids = batches[batch_idx]

            records = [
                dataset[int(idx)]
                for idx in selected_ids
            ]

            batch = collate_graph_records(
                records,
                k=K_PATHS,
                alpha=ALPHA,
                device=DEVICE
            )

            optimizer.zero_grad(set_to_none=True)

            ll = molecular_log_likelihood(
                net, batch
            )

            loss = toxicity_objective(
                ll,
                batch["tox_class"],
                lam=run["config"]["tox_lambda"]
            )

            if not torch.isfinite(loss):
                raise FloatingPointError(
                    f"Non-finite loss at "
                    f"step {run['step'] + 1}"
                )

            loss.backward()

            # Global gradient-norm clipping
            grad_norm = (
                torch.nn.utils.clip_grad_norm_(
                    net.parameters(),
                    max_norm=GRAD_CLIP,
                    error_if_nonfinite=True
                )
            )

            optimizer.step()

            run["step"] += 1
            completed += 1

            # Update exact resume position
            if batch_idx + 1 == len(batches):
                run["epoch"] = epoch + 1
                run["batch_offset"] = 0
            else:
                run["epoch"] = epoch
                run["batch_offset"] = batch_idx + 1

            # Reference-style multiplicative LR decay
            if run["step"] % LR_DECAY_STEP == 0:

                for group in optimizer.param_groups:
                    group["lr"] *= 1.0 - LR_DECAY

            # Log training statistics
            with torch.no_grad():

                y = batch["tox_class"]
                ll_detached = ll.detach()

                ll0 = ll_detached[y == 0]
                ll1 = ll_detached[y == 1]

                record = {
                    "step": run["step"],
                    "epoch": run["epoch"],
                    "loss": float(loss.item()),
                    "grad_norm": float(
                        grad_norm.item()
                    ),
                    "lr": optimizer.param_groups[0]["lr"],
                    "neg_ll_mean": (
                        float(ll0.mean().item())
                        if ll0.numel() else None
                    ),
                    "pos_ll_mean": (
                        float(ll1.mean().item())
                        if ll1.numel() else None
                    ),
                }

            history.append(record)

            if (
                completed % log_every == 0
                or completed == 1
                or completed == num_steps
            ):

                with run["train_log_path"].open(
                    "a", encoding="utf-8"
                ) as f:
                    f.write(json.dumps(record) + "\n")

                print(
                    f"Step {run['step']:>6} | "
                    f"Loss {record['loss']:+.4f} | "
                    f"Grad {record['grad_norm']:.3f} | "
                    f"LR {record['lr']:.6f}"
                )

            if (
                validation_every
                and run["step"] % validation_every == 0
            ):

                metrics = evaluate_training_run(run)

                print(
                    f"Validation | "
                    f"Loss {metrics['val_loss']:+.4f} | "
                    f"Neg LL {metrics['val_neg_ll_mean']} | "
                    f"Pos LL {metrics['val_pos_ll_mean']}"
                )

            if (
                checkpoint_every
                and run["step"] % checkpoint_every == 0
            ):
                save_training_checkpoint(run)

            if completed >= num_steps:
                break

    # Ensure each invocation ends with validation
    # and a resumable checkpoint.
    if not (
        validation_every
        and run["step"] % validation_every == 0
    ):

        metrics = evaluate_training_run(run)

        print(
            f"Final validation | "
            f"Loss {metrics['val_loss']:+.4f}"
        )

    checkpoint_path = save_training_checkpoint(run)

    print(
        "\nCheckpoint:",
        checkpoint_path
    )

    print(
        "Elapsed:",
        round(
            (time.perf_counter() - start_time) / 60,
            2
        ),
        "minutes"
    )

    return history


# ============================================================
# 7. MODEL WEIGHT EXPORT (FOR LATER)
# ============================================================

def export_model_weights(
    run,
    filename="model_final.pt"
):
    """
    Export portable model weights and architecture config.

    Use this after the chosen full training run.
    """

    output_path = run["dir"] / filename

    if output_path.exists():
        raise FileExistsError(
            f"Weight file already exists: {output_path}"
        )

    torch.save(
        {
            "model_state_dict":
                run["model"].state_dict(),
            "model_config": MODEL_CONFIG,
            "atom_types": list(
                get_mol_spec().atom_types
            ),
            "training_config": run["config"],
            "step": run["step"],
        },
        output_path
    )

    return output_path


# ============================================================
# 8. TWO-STEP GPU SMOKE TEST
# ============================================================

print("=" * 65)
print("BLOCK 12: TRAINING FRAMEWORK + SMOKE TEST")
print("=" * 65)

# Select 16 Ames-negative + 16 Ames-positive
# TRAINING molecules, without touching validation.

smoke_train_ids = np.concatenate([
    train_indices[
        labels[train_indices] == 0
    ][:16],
    train_indices[
        labels[train_indices] == 1
    ][:16]
])

# Fixed 8 negative + 8 positive validation molecules
smoke_val_ids = np.concatenate([
    val_eval_indices[
        labels[val_eval_indices] == 0
    ][:8],
    val_eval_indices[
        labels[val_eval_indices] == 1
    ][:8]
])

assert len(smoke_train_ids) == 32
assert len(smoke_val_ids) == 16

smoke_run_name = "block12_smoke_lambda_1"

smoke_checkpoint_path = (
    RUNS_DIR / smoke_run_name / "checkpoint.pt"
)

# Safe reruns: resume an existing smoke checkpoint.
smoke_run = create_training_run(
    run_name=smoke_run_name,
    lam=TOX_LAMBDA,
    learning_rate=LEARNING_RATE,
    train_ids=smoke_train_ids,
    val_ids=smoke_val_ids,
    resume=smoke_checkpoint_path.exists()
)

print("\n[SMOKE TEST CONFIGURATION]")
print("Training molecules:", len(smoke_train_ids))
print("Validation molecules:", len(smoke_val_ids))
print("Lambda:", TOX_LAMBDA)
print("Learning rate:", LEARNING_RATE)
print("Device:", next(smoke_run["model"].parameters()).device)

# Exactly two total optimizer steps.
remaining_steps = max(
    0, 2 - smoke_run["step"]
)

if remaining_steps > 0:

    smoke_history = train_steps(
        smoke_run,
        num_steps=remaining_steps,
        validation_every=2,
        checkpoint_every=2,
        log_every=1
    )

else:
    print("\nSmoke test already completed.")

# ============================================================
# 9. CHECKPOINT INTEGRITY TEST
# ============================================================

print("\n[CHECKPOINT VERIFICATION]")

assert smoke_run["step"] >= 2
assert smoke_checkpoint_path.exists()

saved = torch.load(
    smoke_checkpoint_path,
    map_location="cpu",
    weights_only=False
)

assert saved["step"] == smoke_run["step"]

assert saved["config"] == smoke_run["config"]

assert "optimizer_state_dict" in saved
assert "model_state_dict" in saved

assert torch.equal(
    saved["model_state_dict"]["init_atom_logits"],
    smoke_run["model"].init_atom_logits.detach().cpu()
)

assert smoke_run["best_path"].exists()

print("Resumable checkpoint: PASSED")
print("Optimizer state saved: PASSED")
print("Model weights saved: PASSED")
print("Best validation checkpoint: PASSED")

print("\n[OUTPUT FILES]")
print(smoke_run["dir"])

print("\n" + "=" * 65)
print("BLOCK 12 COMPLETED SUCCESSFULLY!")


BLOCK 12: TRAINING FRAMEWORK + SMOKE TEST
Resumed block12_smoke_lambda_1 at step 2

[SMOKE TEST CONFIGURATION]
Training molecules: 32
Validation molecules: 16
Lambda: 1.0
Learning rate: 0.001
Device: cuda:0

Smoke test already completed.

[CHECKPOINT VERIFICATION]
Resumable checkpoint: PASSED
Optimizer state saved: PASSED
Model weights saved: PASSED
Best validation checkpoint: PASSED

[OUTPUT FILES]
/home/muhamed/repo/CTDDG/REWORKED_CODE/PRETRAINING_OUTPUT/toxicity_aware/runs/block12_smoke_lambda_1

BLOCK 12 COMPLETED SUCCESSFULLY!


## Toxicity Lambda Hyperparameter Tuning

In [14]:

# ============================================================
# BLOCK 13: TOXICITY LAMBDA HYPERPARAMETER SWEEP
# Matched experiments with reproducible training data
# ============================================================

import os
import json
import csv
import gc
import numpy as np
import torch
from pathlib import Path

# ============================================================
# 1. TUNING CONFIGURATION
# ============================================================

TUNE_LAMBDAS = [0.25, 0.50, 1.00, 2.00]

TUNE_TRAIN_SIZE = 4096
TUNE_VAL_SIZE = 256

TUNE_STEPS = 60
TUNE_VAL_EVERY = 20
TUNE_LEARNING_RATE = LEARNING_RATE

TUNE_SEED = SEED + 2027

TUNE_DIR = (
    PRETRAINING_DIR
    / "toxicity_aware"
    / "lambda_tuning"
)

TUNE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 65)
print("BLOCK 13: TOXICITY LAMBDA HYPERPARAMETER TUNING")
print("=" * 65)

# ============================================================
# 2. REPRODUCIBLE STRATIFIED TUNING DATASET
# ============================================================

rng = np.random.default_rng(TUNE_SEED)

# Keep the original training class ratio.
train_negative_pool = train_indices[
    labels[train_indices] == 0
]

train_positive_pool = train_indices[
    labels[train_indices] == 1
]

positive_fraction = (
    len(train_positive_pool)
    / len(train_indices)
)

n_train_positive = round(
    TUNE_TRAIN_SIZE * positive_fraction
)

n_train_negative = (
    TUNE_TRAIN_SIZE - n_train_positive
)

selected_negative = rng.choice(
    train_negative_pool,
    size=n_train_negative,
    replace=False
)

selected_positive = rng.choice(
    train_positive_pool,
    size=n_train_positive,
    replace=False
)

tune_train_ids = np.sort(
    np.concatenate([
        selected_negative,
        selected_positive
    ])
)

# Select a stratified subset of the existing
# fixed validation set from Block 3.
val_negative_pool = val_eval_indices[
    labels[val_eval_indices] == 0
]

val_positive_pool = val_eval_indices[
    labels[val_eval_indices] == 1
]

n_val_positive = round(
    TUNE_VAL_SIZE
    * len(val_positive_pool)
    / len(val_eval_indices)
)

n_val_negative = (
    TUNE_VAL_SIZE - n_val_positive
)

selected_val_negative = rng.choice(
    val_negative_pool,
    size=n_val_negative,
    replace=False
)

selected_val_positive = rng.choice(
    val_positive_pool,
    size=n_val_positive,
    replace=False
)

tune_val_ids = np.concatenate([
    selected_val_negative,
    selected_val_positive
])

# Sort validation by molecular size for
# more memory-efficient graph batching.
tune_val_ids = np.asarray(
    sorted(
        map(int, tune_val_ids),
        key=lambda idx: len(dataset[idx])
    ),
    dtype=np.int64
)

assert len(tune_train_ids) == TUNE_TRAIN_SIZE
assert len(tune_val_ids) == TUNE_VAL_SIZE

assert not np.intersect1d(
    tune_train_ids,
    tune_val_ids
).size

print("\n[TUNING DATASET]")

print("Training molecules:", len(tune_train_ids))
print("Validation molecules:", len(tune_val_ids))

print(
    "Training Ames-negative:",
    int(np.sum(labels[tune_train_ids] == 0))
)

print(
    "Training Ames-positive:",
    int(np.sum(labels[tune_train_ids] == 1))
)

print(
    "Validation Ames-negative:",
    int(np.sum(labels[tune_val_ids] == 0))
)

print(
    "Validation Ames-positive:",
    int(np.sum(labels[tune_val_ids] == 1))
)

# ============================================================
# 3. SAVE TUNING CONFIGURATION
# ============================================================

tune_manifest = {
    "lambdas": TUNE_LAMBDAS,
    "training_size": TUNE_TRAIN_SIZE,
    "validation_size": TUNE_VAL_SIZE,
    "steps_per_run": TUNE_STEPS,
    "validation_every": TUNE_VAL_EVERY,
    "learning_rate": TUNE_LEARNING_RATE,
    "tuning_seed": TUNE_SEED,
    "model_seed": SEED,
    "training_indices_sha256":
        hash_indices(tune_train_ids),
    "validation_indices_sha256":
        hash_indices(tune_val_ids),
    "model_config": MODEL_CONFIG,
    "k_paths": K_PATHS,
    "alpha": ALPHA,
}

manifest_path = TUNE_DIR / "tuning_manifest.json"

if manifest_path.exists():

    previous = json.loads(
        manifest_path.read_text(encoding="utf-8")
    )

    assert previous == tune_manifest, (
        "Existing tuning configuration differs. "
        "Do not mix incompatible experiments."
    )

else:

    manifest_path.write_text(
        json.dumps(tune_manifest, indent=2),
        encoding="utf-8"
    )

# ============================================================
# 4. HELPER: READ VALIDATION HISTORY
# ============================================================

def read_validation_history(path):

    records = {}

    with open(path, "r", encoding="utf-8") as f:

        for line in f:

            if not line.strip():
                continue

            item = json.loads(line)

            # Keep latest entry for each step.
            records[int(item["step"])] = item

    return records


# ============================================================
# 5. RUN MATCHED LAMBDA EXPERIMENTS
# ============================================================

tuning_results = []

for lam in TUNE_LAMBDAS:

    run_name = (
        f"tune_lambda_{lam:.2f}"
        .replace(".", "p")
    )

    run_dir = RUNS_DIR / run_name
    checkpoint_path = run_dir / "checkpoint.pt"

    print("\n" + "=" * 65)
    print(f"EXPERIMENT: LAMBDA = {lam}")
    print("=" * 65)

    resume_existing = checkpoint_path.exists()

    run = create_training_run(
        run_name=run_name,
        lam=lam,
        learning_rate=TUNE_LEARNING_RATE,
        train_ids=tune_train_ids,
        val_ids=tune_val_ids,
        resume=resume_existing
    )

    # --------------------------------------------------------
    # Baseline evaluation before any optimizer updates
    # --------------------------------------------------------

    if run["step"] == 0:

        print("\n[BASELINE VALIDATION]")

        baseline = evaluate_training_run(run)

        print(
            "Initial negative LL:",
            baseline["val_neg_ll_mean"]
        )

        print(
            "Initial positive LL:",
            baseline["val_pos_ll_mean"]
        )

    # --------------------------------------------------------
    # Train until the target optimizer step
    # --------------------------------------------------------

    remaining_steps = max(
        0,
        TUNE_STEPS - run["step"]
    )

    if remaining_steps > 0:

        print("\n[TRAINING]")

        train_steps(
            run,
            num_steps=remaining_steps,
            validation_every=TUNE_VAL_EVERY,
            checkpoint_every=TUNE_VAL_EVERY,
            log_every=10
        )

    else:

        print(
            "Experiment already reached target steps."
        )

    # --------------------------------------------------------
    # Read initial and final validation metrics
    # --------------------------------------------------------

    history = read_validation_history(
        run["val_log_path"]
    )

    assert 0 in history, (
        "Missing baseline validation metrics"
    )

    assert TUNE_STEPS in history, (
        "Missing final validation metrics"
    )

    before = history[0]
    after = history[TUNE_STEPS]

    neg_before = before["val_neg_ll_mean"]
    neg_after = after["val_neg_ll_mean"]

    pos_before = before["val_pos_ll_mean"]
    pos_after = after["val_pos_ll_mean"]

    assert all(
        value is not None
        for value in [
            neg_before,
            neg_after,
            pos_before,
            pos_after
        ]
    )

    # Positive improvement: higher log-likelihood
    # for Ames-negative molecules.
    negative_ll_gain = (
        neg_after - neg_before
    )

    # Positive improvement: lower log-likelihood
    # for Ames-positive molecules.
    positive_ll_suppression = (
        pos_before - pos_after
    )

    result = {
        "lambda": lam,
        "steps": run["step"],
        "negative_ll_before": neg_before,
        "negative_ll_after": neg_after,
        "negative_ll_gain": negative_ll_gain,
        "positive_ll_before": pos_before,
        "positive_ll_after": pos_after,
        "positive_ll_suppression":
            positive_ll_suppression,
        "final_signed_loss": after["val_loss"],
        "run_dir": str(run["dir"])
    }

    tuning_results.append(result)

    print("\n[EXPERIMENT RESULT]")

    print(
        f"Negative LL gain: "
        f"{negative_ll_gain:+.4f}"
    )

    print(
        f"Positive LL suppression: "
        f"{positive_ll_suppression:+.4f}"
    )

    print(
        f"Final signed validation loss: "
        f"{after['val_loss']:+.4f}"
    )

    # Free trial-specific model/optimizer memory.
    del run

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ============================================================
# 6. PARETO-FRONTIER SCREENING
# ============================================================

# Higher negative_ll_gain is better.
# Higher positive_ll_suppression is better.
#
# A run is dominated if another run is at least
# as good on both and strictly better on one.
#
# This is preliminary likelihood-based screening,
# not a final molecule-quality ranking.

for candidate in tuning_results:

    dominated = False

    for other in tuning_results:

        if other is candidate:
            continue

        better_or_equal = (
            other["negative_ll_gain"]
            >= candidate["negative_ll_gain"]
            and
            other["positive_ll_suppression"]
            >= candidate["positive_ll_suppression"]
        )

        strictly_better = (
            other["negative_ll_gain"]
            > candidate["negative_ll_gain"]
            or
            other["positive_ll_suppression"]
            > candidate["positive_ll_suppression"]
        )

        if better_or_equal and strictly_better:
            dominated = True
            break

    candidate["pareto_frontier"] = not dominated


# ============================================================
# 7. SAVE COMPARISON RESULTS
# ============================================================

results_json = TUNE_DIR / "lambda_sweep_results.json"
results_csv = TUNE_DIR / "lambda_sweep_results.csv"

results_json.write_text(
    json.dumps(tuning_results, indent=2),
    encoding="utf-8"
)

with open(
    results_csv,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=list(tuning_results[0].keys())
    )

    writer.writeheader()
    writer.writerows(tuning_results)


# ============================================================
# 8. FINAL COMPARISON
# ============================================================

print("\n" + "=" * 80)
print("LAMBDA TUNING SUMMARY")
print("=" * 80)

print(
    f"{'Lambda':>8} | "
    f"{'Neg LL gain':>13} | "
    f"{'Pos suppression':>17} | "
    f"{'Pareto':>8}"
)

print("-" * 65)

for item in tuning_results:

    print(
        f"{item['lambda']:>8.2f} | "
        f"{item['negative_ll_gain']:>+13.4f} | "
        f"{item['positive_ll_suppression']:>+17.4f} | "
        f"{str(item['pareto_frontier']):>8}"
    )

pareto_lambdas = [
    item["lambda"]
    for item in tuning_results
    if item["pareto_frontier"]
]

print("\nPareto-frontier lambdas:", pareto_lambdas)

print("\nSaved files:")
print(results_json)
print(results_csv)
print(manifest_path)

print("\nIMPORTANT:")
print(
    "This is an initial likelihood-based screening. "
    "Do not select the final lambda solely from "
    "signed losses or these short experiments."
)

print("\nBLOCK 13 COMPLETED SUCCESSFULLY!")


BLOCK 13: TOXICITY LAMBDA HYPERPARAMETER TUNING

[TUNING DATASET]
Training molecules: 4096
Validation molecules: 256
Training Ames-negative: 2935
Training Ames-positive: 1161
Validation Ames-negative: 184
Validation Ames-positive: 72

EXPERIMENT: LAMBDA = 0.25
Resumed tune_lambda_0p25 at step 60
Experiment already reached target steps.

[EXPERIMENT RESULT]
Negative LL gain: +214.2266
Positive LL suppression: -211.8913
Final signed validation loss: +36.9963

EXPERIMENT: LAMBDA = 0.5
Resumed tune_lambda_0p50 at step 60
Experiment already reached target steps.

[EXPERIMENT RESULT]
Negative LL gain: +205.4921
Positive LL suppression: -201.8234
Final signed validation loss: +37.4510

EXPERIMENT: LAMBDA = 1.0
Resumed tune_lambda_1p00 at step 60
Experiment already reached target steps.

[EXPERIMENT RESULT]
Negative LL gain: +179.7893
Positive LL suppression: -160.9555
Final signed validation loss: +34.2002

EXPERIMENT: LAMBDA = 2.0
Resumed tune_lambda_2p00 at step 60
Experiment already reache

## Training speed benchmark

In [15]:

# ============================================================
# BLOCK 14: CTDDG FULL-TRAINING RUNTIME BENCHMARK
# No full training is started in this block.
# ============================================================

import time
import math
import json
import gc
import random
import numpy as np
import torch

# ------------------------------------------------------------
# 1. Benchmark configuration
# ------------------------------------------------------------

BENCH_WARMUP = 8
BENCH_STEPS = 64
BENCH_POOL_SIZE = 2048
BENCH_LAMBDA = 2.0  # For timing only, not final selection

PLANNED_EPOCHS = EPOCHS
PLANNED_VAL_EVERY = 2000
PLANNED_SAVE_EVERY = 500

BENCH_REPORT = (
    PRETRAINING_DIR
    / "toxicity_aware"
    / "training_benchmark.json"
)

print("=" * 65)
print("BLOCK 14: FULL TRAINING RUNTIME BENCHMARK")
print("=" * 65)

print("Device:", DEVICE)
print("Batch size:", BATCH_SIZE)
print("Paths per molecule:", K_PATHS)
print("Warm-up steps:", BENCH_WARMUP)
print("Measured steps:", BENCH_STEPS)

# ------------------------------------------------------------
# 2. Select representative training molecules
# ------------------------------------------------------------

rng = np.random.default_rng(SEED + 1400)

bench_ids = rng.choice(
    train_indices,
    size=BENCH_POOL_SIZE,
    replace=False
)

bench_sorted_ids = sorted(
    map(int, bench_ids),
    key=lambda idx: len(dataset[idx])
)

bench_batches = make_epoch_batches(
    bench_sorted_ids,
    epoch=0,
    batch_size=BATCH_SIZE,
    seed=SEED
)

assert len(bench_batches) >= (
    BENCH_WARMUP + BENCH_STEPS
)

# ------------------------------------------------------------
# 3. Create isolated benchmark model
# ------------------------------------------------------------

bench_model = make_fresh_model(
    seed=SEED
)

bench_optimizer = torch.optim.Adam(
    bench_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

bench_model.train()

model_device = next(
    bench_model.parameters()
).device

if model_device.type == "cuda":
    torch.cuda.reset_peak_memory_stats(model_device)

def synchronize_gpu():
    if model_device.type == "cuda":
        torch.cuda.synchronize(model_device)

# ------------------------------------------------------------
# 4. Measure real training steps
# ------------------------------------------------------------

step_times = []
step_losses = []

python_rng_state = random.getstate()

try:
    random.seed(SEED + 1401)

    total_bench_steps = BENCH_WARMUP + BENCH_STEPS

    for step_idx in range(total_bench_steps):

        synchronize_gpu()
        start = time.perf_counter()

        selected_ids = bench_batches[step_idx]

        records = [
            dataset[int(idx)]
            for idx in selected_ids
        ]

        batch = collate_graph_records(
            records,
            k=K_PATHS,
            alpha=ALPHA,
            device=model_device
        )

        bench_optimizer.zero_grad(
            set_to_none=True
        )

        ll = molecular_log_likelihood(
            bench_model,
            batch
        )

        loss = toxicity_objective(
            ll,
            batch["tox_class"],
            lam=BENCH_LAMBDA
        )

        if not torch.isfinite(loss):
            raise FloatingPointError(
                f"Non-finite benchmark loss at "
                f"step {step_idx + 1}"
            )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            bench_model.parameters(),
            max_norm=GRAD_CLIP,
            error_if_nonfinite=True
        )

        bench_optimizer.step()

        synchronize_gpu()
        elapsed = time.perf_counter() - start

        # Do not include initial CUDA warm-up.
        if step_idx >= BENCH_WARMUP:

            step_times.append(elapsed)
            step_losses.append(
                float(loss.item())
            )

        if (step_idx + 1) % 16 == 0:

            print(
                f"Completed {step_idx + 1}/"
                f"{total_bench_steps} steps"
            )

finally:
    random.setstate(python_rng_state)

# ------------------------------------------------------------
# 5. Benchmark validation on 512 molecules
# ------------------------------------------------------------

print("\n[BENCHMARKING VALIDATION]")

bench_model.eval()

val_ids = sorted(
    map(int, val_eval_indices),
    key=lambda idx: len(dataset[idx])
)

validation_rng = random.getstate()
val_count = 0

try:
    random.seed(VAL_SEED + 12345)

    synchronize_gpu()
    val_start = time.perf_counter()

    with torch.no_grad():

        for offset in range(
            0, len(val_ids), BATCH_SIZE
        ):

            selected = val_ids[
                offset:offset + BATCH_SIZE
            ]

            records = [
                dataset[idx]
                for idx in selected
            ]

            batch = collate_graph_records(
                records,
                k=K_PATHS,
                alpha=ALPHA,
                device=model_device
            )

            ll = molecular_log_likelihood(
                bench_model, batch
            )

            assert torch.isfinite(ll).all()

            val_count += len(records)

    synchronize_gpu()

    validation_seconds = (
        time.perf_counter() - val_start
    )

finally:
    random.setstate(validation_rng)

assert val_count == len(val_ids)

# ------------------------------------------------------------
# 6. Calculate realistic training estimates
# ------------------------------------------------------------

mean_step = float(np.mean(step_times))
median_step = float(np.median(step_times))
p90_step = float(np.percentile(step_times, 90))

# Matches the bucket construction used in Block 12.
bucket_size = max(BATCH_SIZE * 32, 256)

assert bucket_size % BATCH_SIZE == 0

n_full_buckets, remainder = divmod(
    len(train_indices),
    bucket_size
)

steps_per_epoch = (
    n_full_buckets * (bucket_size // BATCH_SIZE)
    + remainder // BATCH_SIZE
)

total_training_steps = (
    steps_per_epoch * PLANNED_EPOCHS
)

training_seconds = (
    total_training_steps * mean_step
)

n_validations = math.ceil(
    total_training_steps / PLANNED_VAL_EVERY
)

validation_overhead = (
    n_validations * validation_seconds
)

# Checkpoint writing and scheduler overhead are
# not included; actual runtime may be longer.
estimated_total_seconds = (
    training_seconds + validation_overhead
)

mean_effective_step = (
    mean_step
    + validation_seconds / PLANNED_VAL_EVERY
)

four_hour_steps = int(
    (4 * 3600) / mean_effective_step
)

four_hour_epochs = (
    four_hour_steps / steps_per_epoch
)

gpu_peak_gb = None

if model_device.type == "cuda":

    gpu_peak_gb = (
        torch.cuda.max_memory_allocated(
            model_device
        ) / 1024**3
    )

# ------------------------------------------------------------
# 7. Save benchmark results
# ------------------------------------------------------------

report = {
    "batch_size": BATCH_SIZE,
    "k_paths": K_PATHS,
    "benchmark_lambda": BENCH_LAMBDA,
    "warmup_steps": BENCH_WARMUP,
    "measured_steps": BENCH_STEPS,
    "mean_step_seconds": mean_step,
    "median_step_seconds": median_step,
    "p90_step_seconds": p90_step,
    "validation_molecules": val_count,
    "validation_seconds": validation_seconds,
    "training_molecules": len(train_indices),
    "epochs": PLANNED_EPOCHS,
    "steps_per_epoch": steps_per_epoch,
    "total_steps": total_training_steps,
    "planned_validation_every": PLANNED_VAL_EVERY,
    "planned_checkpoint_every": PLANNED_SAVE_EVERY,
    "estimated_training_hours": training_seconds / 3600,
    "estimated_validation_hours": validation_overhead / 3600,
    "estimated_total_hours": estimated_total_seconds / 3600,
    "estimated_steps_in_four_hours": four_hour_steps,
    "estimated_epochs_in_four_hours": four_hour_epochs,
    "gpu_peak_gb": gpu_peak_gb,
    "notes": (
        "Approximation based on an isolated Jupyter GPU "
        "benchmark. Excludes checkpoint I/O, scheduler "
        "delays, cluster contention and job startup."
    )
}

BENCH_REPORT.parent.mkdir(
    parents=True,
    exist_ok=True
)

BENCH_REPORT.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

# ------------------------------------------------------------
# 8. Display summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("GPU BENCHMARK RESULTS")
print("=" * 65)

print(f"Mean step time:   {mean_step:.4f} seconds")
print(f"Median step time: {median_step:.4f} seconds")
print(f"P90 step time:    {p90_step:.4f} seconds")

print(
    f"\n512-molecule validation: "
    f"{validation_seconds:.2f} seconds"
)

if gpu_peak_gb is not None:
    print(
        f"GPU peak allocated memory: "
        f"{gpu_peak_gb:.2f} GB"
    )

print("\n[FULL DATASET PROJECTION]")

print(f"Molecules: {len(train_indices):,}")
print(f"Steps per epoch: {steps_per_epoch:,}")
print(f"Epochs: {PLANNED_EPOCHS}")
print(f"Total steps: {total_training_steps:,}")

print(
    f"\nEstimated training time: "
    f"{training_seconds / 3600:.2f} hours"
)

print(
    f"Estimated validation overhead: "
    f"{validation_overhead / 3600:.2f} hours"
)

print(
    f"Estimated total (before checkpoint overhead): "
    f"{estimated_total_seconds / 3600:.2f} hours"
)

print("\n[FOUR-HOUR BUDGET]")

print(
    f"Estimated optimizer steps: "
    f"{four_hour_steps:,}"
)

print(
    f"Estimated epochs completed: "
    f"{four_hour_epochs:.3f}"
)

print("\nReport saved:", BENCH_REPORT)

# ------------------------------------------------------------
# 9. Clean benchmark GPU memory
# ------------------------------------------------------------

del bench_model
del bench_optimizer
del batch
del ll
del loss

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nBLOCK 14 COMPLETED SUCCESSFULLY!")


BLOCK 14: FULL TRAINING RUNTIME BENCHMARK
Device: cuda
Batch size: 8
Paths per molecule: 5
Warm-up steps: 8
Measured steps: 64
Completed 16/72 steps
Completed 32/72 steps
Completed 48/72 steps
Completed 64/72 steps

[BENCHMARKING VALIDATION]

GPU BENCHMARK RESULTS
Mean step time:   0.0414 seconds
Median step time: 0.0414 seconds
P90 step time:    0.0509 seconds

512-molecule validation: 1.63 seconds
GPU peak allocated memory: 2.02 GB

[FULL DATASET PROJECTION]
Molecules: 1,035,981
Steps per epoch: 129,497
Epochs: 5
Total steps: 647,485

Estimated training time: 7.45 hours
Estimated validation overhead: 0.15 hours
Estimated total (before checkpoint overhead): 7.60 hours

[FOUR-HOUR BUDGET]
Estimated optimizer steps: 340,804
Estimated epochs completed: 2.632

Report saved: /home/muhamed/repo/CTDDG/REWORKED_CODE/PRETRAINING_OUTPUT/toxicity_aware/training_benchmark.json

BLOCK 14 COMPLETED SUCCESSFULLY!


## MAIN CTDDG PRETRAINING

In [16]:

# ============================================================
# BLOCK 15: MAIN CTDDG PRETRAINING
# JUPYTER VERSION — BATCH SIZE 32
# ============================================================

import time
import gc
import torch
import numpy as np

# ------------------------------------------------------------
# 1. Training configuration
# ------------------------------------------------------------

BATCH_SIZE = 32

MAIN_LAMBDA = 2.0
MAIN_LEARNING_RATE = 1e-3
MAIN_EPOCHS = 5

CHECKPOINT_EVERY = 250
VALIDATION_EVERY = 2000
LOG_EVERY = 100

# Stop cleanly before a 4-hour session budget.
MAX_HOURS = 3.70

# Run in chunks so we can check elapsed time.
CHUNK_STEPS = 1000

RUN_NAME = "ctddg_main_bs32_lambda2"

# ------------------------------------------------------------
# 2. Check batch-size-32 GPU compatibility
# ------------------------------------------------------------

print("=" * 65)
print("CTDDG MAIN TRAINING — BATCH SIZE 32")
print("=" * 65)

print("\n[GPU PREFLIGHT]")

# Select a small mixture of both toxicity classes.
preflight_ids = np.concatenate([
    train_indices[labels[train_indices] == 0][:24],
    train_indices[labels[train_indices] == 1][:8]
])

preflight_records = [
    dataset[int(i)] for i in preflight_ids
]

test32 = collate_graph_records(
    preflight_records,
    k=K_PATHS,
    alpha=ALPHA,
    device=DEVICE
)

preflight_model = make_fresh_model(SEED)
preflight_model.train()

preflight_model.zero_grad(set_to_none=True)

preflight_ll = molecular_log_likelihood(
    preflight_model,
    test32
)

preflight_loss = toxicity_objective(
    preflight_ll,
    test32["tox_class"],
    lam=MAIN_LAMBDA
)

assert torch.isfinite(preflight_loss)

preflight_loss.backward()

torch.nn.utils.clip_grad_norm_(
    preflight_model.parameters(),
    max_norm=GRAD_CLIP,
    error_if_nonfinite=True
)

print("Batch size:", BATCH_SIZE)
print("Graph paths:", BATCH_SIZE * K_PATHS)
print("Loss:", float(preflight_loss.item()))
print("Forward/backward: PASSED")

if torch.cuda.is_available():
    print(
        "GPU peak memory:",
        f"{torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
    )

del preflight_model
del test32
del preflight_ll
del preflight_loss

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ------------------------------------------------------------
# 3. Initialize or resume main training
# ------------------------------------------------------------

main_checkpoint_path = (
    RUNS_DIR / RUN_NAME / "checkpoint.pt"
)

main_run = create_training_run(
    run_name=RUN_NAME,
    lam=MAIN_LAMBDA,
    learning_rate=MAIN_LEARNING_RATE,
    train_ids=train_indices,
    val_ids=val_eval_indices,
    resume=main_checkpoint_path.exists()
)

steps_per_epoch = len(
    make_epoch_batches(
        main_run["sorted_ids"],
        epoch=0,
        batch_size=BATCH_SIZE,
        seed=SEED
    )
)

total_steps = MAIN_EPOCHS * steps_per_epoch

print("\n[TRAINING CONFIGURATION]")
print("Training molecules:", len(train_indices))
print("Batch size:", BATCH_SIZE)
print("Steps per epoch:", steps_per_epoch)
print("Total steps:", total_steps)
print("Current completed steps:", main_run["step"])
print("Lambda:", MAIN_LAMBDA)
print("Learning rate:", MAIN_LEARNING_RATE)

print("\nCheckpoint folder:")
print(main_run["dir"])

# ------------------------------------------------------------
# 4. Main training loop
# ------------------------------------------------------------

session_start = time.monotonic()
max_seconds = MAX_HOURS * 3600

while main_run["step"] < total_steps:

    elapsed = time.monotonic() - session_start

    if elapsed >= max_seconds:
        print("\nSESSION TIME BUDGET REACHED")
        print("Saving checkpoint...")
        save_training_checkpoint(main_run)
        break

    remaining = total_steps - main_run["step"]

    steps_this_chunk = min(
        CHUNK_STEPS,
        remaining
    )

    print("\n" + "-" * 65)
    print(
        f"Training step {main_run['step'] + 1:,}"
        f" to {main_run['step'] + steps_this_chunk:,}"
    )
    print("-" * 65)

    train_steps(
        main_run,
        num_steps=steps_this_chunk,
        validation_every=VALIDATION_EVERY,
        checkpoint_every=CHECKPOINT_EVERY,
        log_every=LOG_EVERY
    )

    completed_epochs = (
        main_run["step"] / steps_per_epoch
    )

    elapsed_hours = (
        time.monotonic() - session_start
    ) / 3600

    print(
        f"\nProgress: {main_run['step']:,}"
        f"/{total_steps:,} steps"
    )

    print(
        f"Epoch equivalent: "
        f"{completed_epochs:.3f}/{MAIN_EPOCHS}"
    )

    print(
        f"Session elapsed: {elapsed_hours:.2f} hours"
    )

# ------------------------------------------------------------
# 5. Save final checkpoint
# ------------------------------------------------------------

save_training_checkpoint(main_run)

print("\n" + "=" * 65)
print("TRAINING SESSION SUMMARY")
print("=" * 65)

print("Completed steps:", main_run["step"])
print("Target steps:", total_steps)

print(
    "Completed epochs:",
    round(main_run["step"] / steps_per_epoch, 3)
)

print(
    "Elapsed hours:",
    round(
        (time.monotonic() - session_start) / 3600,
        3
    )
)

print("Checkpoint:", main_run["checkpoint_path"])
print("Best checkpoint:", main_run["best_path"])

# ------------------------------------------------------------
# 6. Export final weights if training is complete
# ------------------------------------------------------------

if main_run["step"] >= total_steps:

    final_weights = (
        main_run["dir"] / "model_final.pt"
    )

    if not final_weights.exists():

        export_model_weights(
            main_run,
            filename="model_final.pt"
        )

    print("\nFULL PRETRAINING COMPLETED!")
    print("Final model weights:", final_weights)

else:

    print("\nTRAINING PAUSED SAFELY.")
    print("The latest checkpoint has been saved.")
    print("Rerun this cell to resume training.")

print("\nBLOCK 15 SESSION COMPLETED")


CTDDG MAIN TRAINING — BATCH SIZE 32

[GPU PREFLIGHT]
Batch size: 32
Graph paths: 160
Loss: 44.520606994628906
Forward/backward: PASSED
GPU peak memory: 4.71 GB
Resumed ctddg_main_bs32_lambda2 at step 119750

[TRAINING CONFIGURATION]
Training molecules: 1035981
Batch size: 32
Steps per epoch: 32374
Total steps: 161870
Current completed steps: 119750
Lambda: 2.0
Learning rate: 0.001

Checkpoint folder:
/home/muhamed/repo/CTDDG/REWORKED_CODE/PRETRAINING_OUTPUT/toxicity_aware/runs/ctddg_main_bs32_lambda2

-----------------------------------------------------------------
Training step 119,751 to 120,750
-----------------------------------------------------------------
Step 119751 | Loss -28.3683 | Grad 181.454 | LR 0.000302
Step 119850 | Loss -114.0937 | Grad 135.767 | LR 0.000302
Step 119950 | Loss -296.3348 | Grad 310.566 | LR 0.000301
Validation | Loss -122.3075 | Neg LL -138.66424144776056 | Pos LL -391.41798895474136
Step 120050 | Loss -143.7475 | Grad 256.875 | LR 0.000301


Step 120150 | Loss -28.5437 | Grad 392.630 | LR 0.000301
Step 120250 | Loss -178.8832 | Grad 122.366 | LR 0.000300
Step 120350 | Loss -159.3628 | Grad 341.501 | LR 0.000300
Step 120450 | Loss -384.7931 | Grad 271.900 | LR 0.000300
Step 120550 | Loss -136.5838 | Grad 439.892 | LR 0.000300
Step 120650 | Loss -46.3124 | Grad 317.918 | LR 0.000299
Step 120750 | Loss -3.7422 | Grad 46.848 | LR 0.000299
Final validation | Loss -113.9923

Checkpoint: /home/muhamed/repo/CTDDG/REWORKED_CODE/PRETRAINING_OUTPUT/toxicity_aware/runs/ctddg_main_bs32_lambda2/checkpoint.pt
Elapsed: 2.5 minutes

Progress: 120,750/161,870 steps
Epoch equivalent: 3.730/5
Session elapsed: 0.04 hours

-----------------------------------------------------------------
Training step 120,751 to 121,750
-----------------------------------------------------------------
Step 120751 | Loss -74.7172 | Grad 353.576 | LR 0.000299
Step 120850 | Loss -39.8533 | Grad 471.107 | LR 0.000299
Step 120950 | Loss -123.0632 | Grad 341.028 | LR

## TRAINING VERIFICATION AND ANALYSIS